# Unlearning Project — Post-Hoc Audit, Adjudication, and Corrected Selection v1.3.1

**Analysis protocol:** `prelangchain_posthoc_v1_3_1`  
**Frozen source protocol:** `prelangchain_ab_v1_2_api_hardened`  
**Purpose:** preserve the completed paid experiment, import it from an immutable Drive snapshot, correct analysis defects, prepare human adjudication, and rescore the existing predictions without new model calls.

This notebook is deliberately **analysis-only**:

- it contains no provider adapters;
- it never imports OpenAI, Anthropic, or Gemini SDKs;
- it never calls a hosted model;
- it never modifies raw source logs or the original selection JSON files;
- all corrected products are written to a separate analysis directory.

The source experiment already retained exact requests, raw responses, parsed outputs, token/cost metadata, model identifiers, and deterministic run keys. This notebook makes an additional frozen, SHA-256-verified copy before analysis.

## What this notebook fixes

1. Freezes all important source outputs in a versioned Google Drive folder.
2. Imports raw phase results from that frozen snapshot rather than from live run directories.
3. Separates the **prompt-input hash** from the **human-gold-label hash**.
4. Replaces substring-based EPA detection with an exact normalized document identity.
5. Re-audits evidence with strict, typography-normalized, scope-aware failure categories.
6. Audits the historical target taxonomy before reporting target accuracy.
7. Calculates Krippendorff's alpha and the other stability/reliability measures.
8. Reconstructs Phase 1 completeness independently of `ACTIVE_PHASES`.
9. Labels selections honestly as final, no-eligible-winner, or provisional minimum-shortfall candidates.
10. Generates an editable adjudication workbook and automatically rescales all saved predictions when completed labels are supplied.
11. Produces a corrected report and a small confirmatory-experiment plan without launching that experiment.

## Execution map

Run from top to bottom.

- On the first run, the snapshot section copies and verifies the completed v1.2 outputs.
- On later runs, the snapshot is read-only and verified against its lock manifest.
- The adjudication template is generated near the end.
- After humans complete it, save a copy as `adjudication_completed.xlsx` in the displayed adjudication folder and rerun this notebook. No API calls are needed for rescoring.

## 0. Dependency bootstrap

In [1]:
# Analysis-only dependencies. Provider SDKs are intentionally absent.
import importlib.util
import subprocess
import sys

PACKAGE_IMPORTS = {
    "numpy": "numpy",
    "pandas": "pandas",
    "openpyxl": "openpyxl",
    "sklearn": "scikit-learn",
    "scipy": "scipy",
    "statsmodels": "statsmodels",
}

missing_packages = [
    package_name
    for import_name, package_name in PACKAGE_IMPORTS.items()
    if importlib.util.find_spec(import_name) is None
]

if missing_packages:
    print("Installing missing analysis packages:", missing_packages)
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        *missing_packages,
    ])
else:
    print("All analysis dependencies are already installed.")

All analysis dependencies are already installed.


## 1. Imports and immutable analysis contract

In [2]:
from __future__ import annotations

from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable, Optional, Sequence
from difflib import SequenceMatcher
import contextlib
import hashlib
import importlib.metadata
import itertools
import json
import math
import os
import re
import shutil
import sys
import unicodedata

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import binomtest
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    cohen_kappa_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
)
try:
    import krippendorff as krippendorff_package
except ImportError:
    krippendorff_package = None

pd.set_option("display.max_columns", 160)
pd.set_option("display.max_colwidth", 180)
pd.set_option("display.width", 220)

ANALYSIS_VERSION = "prelangchain_posthoc_v1_3_1"
SOURCE_PROTOCOL = "prelangchain_ab_v1_2_api_hardened"
RUN_API_CALLS = False

assert RUN_API_CALLS is False, "This notebook must remain analysis-only."

print({
    "analysis_version": ANALYSIS_VERSION,
    "source_protocol": SOURCE_PROTOCOL,
    "run_api_calls": RUN_API_CALLS,
    "python": sys.version.split()[0],
})

{'analysis_version': 'prelangchain_posthoc_v1_3_1', 'source_protocol': 'prelangchain_ab_v1_2_api_hardened', 'run_api_calls': False, 'python': '3.12.13'}


## 2. Mount Drive and configure source, snapshot, and analysis paths

In [3]:
IN_COLAB = importlib.util.find_spec("google.colab") is not None

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

DEFAULT_SOURCE_ROOT = (
    Path("/content/drive/MyDrive/Unlearning_Project/prelangchain_ab_v1_2_workspace")
    if IN_COLAB
    else Path("/mnt/data/unlearning_posthoc_source_workspace")
)

SOURCE_PROJECT_ROOT = Path(
    os.getenv("UNLEARNING_SOURCE_PROJECT_ROOT", str(DEFAULT_SOURCE_ROOT))
).expanduser().resolve()

SNAPSHOT_NAME = os.getenv(
    "UNLEARNING_SNAPSHOT_NAME",
    "prelangchain_v12_discovery_r1_complete",
).strip()

SNAPSHOT_ROOT = SOURCE_PROJECT_ROOT / "frozen_runs" / SNAPSHOT_NAME
SNAPSHOT_SOURCE_ROOT = SNAPSHOT_ROOT / "source"

ANALYSIS_ROOT = Path(
    os.getenv(
        "UNLEARNING_POSTHOC_ROOT",
        str(SOURCE_PROJECT_ROOT / "analysis" / ANALYSIS_VERSION),
    )
).expanduser().resolve()

TABLES_ROOT = ANALYSIS_ROOT / "tables"
ADJUDICATION_ROOT = ANALYSIS_ROOT / "adjudication"
MANIFEST_ROOT = ANALYSIS_ROOT / "manifests"
REPORTS_ROOT = ANALYSIS_ROOT / "reports"

for folder in [ANALYSIS_ROOT, TABLES_ROOT, ADJUDICATION_ROOT, MANIFEST_ROOT, REPORTS_ROOT]:
    folder.mkdir(parents=True, exist_ok=True)

SOURCE_NOTEBOOK_PATH = os.getenv("UNLEARNING_SOURCE_NOTEBOOK", "").strip()
EPA_DOCUMENT_TITLE = os.getenv(
    "UNLEARNING_EPA_DOCUMENT_TITLE",
    "Lessons Learned: EPA’s Response to Hurricane Katrina",
)

ADJUDICATION_COMPLETED_PATH = Path(
    os.getenv(
        "UNLEARNING_ADJUDICATION_COMPLETED",
        str(ADJUDICATION_ROOT / "adjudication_completed.xlsx"),
    )
).expanduser().resolve()

SPECIFICITY_FLOOR = float(os.getenv("UNLEARNING_SPECIFICITY_FLOOR", "0.80"))
MIN_SCHEMA_VALID_RATE = float(os.getenv("UNLEARNING_MIN_SCHEMA_VALID_RATE", "0.99"))
MIN_EVIDENCE_VALID_RATE = float(os.getenv("UNLEARNING_MIN_EVIDENCE_VALID_RATE", "0.98"))
EXPECTED_PROVIDERS = ("openai", "anthropic", "gemini")
EXPECTED_STABILITY_SEEDS = (17, 43, 101, 211, 307)
STRICT_SOURCE_COMPLETENESS = os.getenv(
    "UNLEARNING_STRICT_SOURCE_COMPLETENESS", "true"
).lower() == "true"

CONFIGURATION = {
    "analysis_version": ANALYSIS_VERSION,
    "source_protocol": SOURCE_PROTOCOL,
    "source_project_root": str(SOURCE_PROJECT_ROOT),
    "snapshot_root": str(SNAPSHOT_ROOT),
    "analysis_root": str(ANALYSIS_ROOT),
    "adjudication_completed_path": str(ADJUDICATION_COMPLETED_PATH),
    "epa_document_title": EPA_DOCUMENT_TITLE,
    "run_api_calls": RUN_API_CALLS,
    "specificity_floor": SPECIFICITY_FLOOR,
    "minimum_schema_valid_rate": MIN_SCHEMA_VALID_RATE,
    "minimum_evidence_valid_rate": MIN_EVIDENCE_VALID_RATE,
}

display(pd.DataFrame([CONFIGURATION]).T.rename(columns={0: "value"}))

Mounted at /content/drive


,value
analysis_version,prelangchain_posthoc_v1_3_1
source_protocol,prelangchain_ab_v1_2_api_hardened
source_project_root,/content/drive/MyDrive/Unlearning_Project/prelangchain_ab_v1_2_workspace
snapshot_root,/content/drive/MyDrive/Unlearning_Project/prelangchain_ab_v1_2_workspace/frozen_runs/prelangchain_v12_discovery_r1_complete
analysis_root,/content/drive/MyDrive/Unlearning_Project/prelangchain_ab_v1_2_workspace/analysis/prelangchain_posthoc_v1_3_1
adjudication_completed_path,/content/drive/MyDrive/Unlearning_Project/prelangchain_ab_v1_2_workspace/analysis/prelangchain_posthoc_v1_3_1/adjudication/adjudication_completed.xlsx
epa_document_title,Lessons Learned: EPA’s Response to Hurricane Katrina
run_api_calls,False
specificity_floor,0.8
minimum_schema_valid_rate,0.99


## 3. Freeze all important source outputs

The snapshot contains the protocol-specific `runs`, `artifacts`, `reports`, `configs`, and `backups` trees, the benchmark workbook, and any located executed notebook. The snapshot is locked only after every copied file is hash-verified.

If a locked snapshot already exists, this section verifies it and does not modify it. To freeze a later source state, choose a new `SNAPSHOT_NAME` rather than overwriting the old one.

In [4]:
def utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


def sha256_text(text: Any) -> str:
    return hashlib.sha256(str(text).encode("utf-8")).hexdigest()


def canonical_json(value: Any) -> str:
    return json.dumps(
        value,
        sort_keys=True,
        ensure_ascii=False,
        separators=(",", ":"),
        default=str,
    )


def atomic_write_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + f".partial-{os.getpid()}")
    temporary.write_text(text, encoding="utf-8")
    os.replace(temporary, path)


def atomic_write_dataframe(frame: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + f".partial-{os.getpid()}")
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


def verified_atomic_copy(source: Path, destination: Path) -> dict[str, Any]:
    source = source.resolve()
    destination = destination.resolve()
    destination.parent.mkdir(parents=True, exist_ok=True)

    source_hash_before = sha256_file(source)
    source_size = source.stat().st_size

    if destination.exists():
        destination_hash = sha256_file(destination)
        if destination_hash == source_hash_before and destination.stat().st_size == source_size:
            return {
                "status": "unchanged",
                "source": str(source),
                "destination": str(destination),
                "size_bytes": source_size,
                "sha256": source_hash_before,
            }

    temporary = destination.with_name(destination.name + f".partial-{os.getpid()}")
    if temporary.exists():
        temporary.unlink()
    shutil.copy2(source, temporary)

    temporary_hash = sha256_file(temporary)
    source_hash_after = sha256_file(source)
    if source_hash_before != source_hash_after:
        temporary.unlink(missing_ok=True)
        raise RuntimeError(
            f"Source changed while being copied: {source}. "
            "Run the snapshot after the source notebook is idle."
        )
    if temporary_hash != source_hash_before:
        temporary.unlink(missing_ok=True)
        raise IOError(f"Hash mismatch while copying {source} to {destination}.")

    os.replace(temporary, destination)
    if sha256_file(destination) != source_hash_before:
        raise IOError(f"Post-copy hash mismatch: {destination}")

    return {
        "status": "copied",
        "source": str(source),
        "destination": str(destination),
        "size_bytes": source_size,
        "sha256": source_hash_before,
    }


def iter_snapshot_files(root: Path) -> Iterable[Path]:
    if not root.exists():
        return []
    return (
        path
        for path in sorted(root.rglob("*"))
        if path.is_file()
        and ".partial-" not in path.name
        and not path.name.endswith((".tmp", ".lock"))
    )

In [5]:
def locate_source_workbook() -> Path:
    candidates = [
        SOURCE_PROJECT_ROOT / "data" / "Unlearning_Codebook_Local_Context_Test_Set.xlsx",
        SOURCE_PROJECT_ROOT / "Unlearning_Codebook_Local_Context_Test_Set.xlsx",
    ]
    candidates.extend(
        SOURCE_PROJECT_ROOT.rglob("Unlearning_Codebook_Local_Context_Test_Set.xlsx")
        if SOURCE_PROJECT_ROOT.exists()
        else []
    )
    for candidate in candidates:
        if candidate.is_file() and SNAPSHOT_ROOT not in candidate.parents:
            return candidate.resolve()
    raise FileNotFoundError(
        "Could not locate Unlearning_Codebook_Local_Context_Test_Set.xlsx below "
        f"{SOURCE_PROJECT_ROOT}."
    )


def locate_source_notebooks() -> list[Path]:
    candidates: list[Path] = []
    if SOURCE_NOTEBOOK_PATH:
        explicit = Path(SOURCE_NOTEBOOK_PATH).expanduser().resolve()
        if explicit.is_file():
            candidates.append(explicit)
    if SOURCE_PROJECT_ROOT.exists():
        patterns = [
            "*PHASE2_RESUME_ORGANIZED*.ipynb",
            "*API_HARDENED*.ipynb",
        ]
        for pattern in patterns:
            candidates.extend(SOURCE_PROJECT_ROOT.rglob(pattern))
    unique: dict[str, Path] = {}
    for path in candidates:
        resolved = path.resolve()
        if SNAPSHOT_ROOT not in resolved.parents:
            unique[str(resolved)] = resolved
    return sorted(unique.values(), key=lambda path: path.stat().st_mtime, reverse=True)


SOURCE_WORKBOOK = locate_source_workbook()
SOURCE_NOTEBOOKS = locate_source_notebooks()

SOURCE_TREES = {
    "runs": SOURCE_PROJECT_ROOT / "runs" / SOURCE_PROTOCOL,
    "artifacts": SOURCE_PROJECT_ROOT / "artifacts" / SOURCE_PROTOCOL,
    "reports": SOURCE_PROJECT_ROOT / "reports" / SOURCE_PROTOCOL,
    "configs": SOURCE_PROJECT_ROOT / "configs" / SOURCE_PROTOCOL,
    "backups": SOURCE_PROJECT_ROOT / "backups" / SOURCE_PROTOCOL,
}

SOURCE_INVENTORY = []
for category, source_root in SOURCE_TREES.items():
    SOURCE_INVENTORY.append({
        "category": category,
        "source_root": str(source_root),
        "exists": source_root.exists(),
        "files": sum(1 for _ in iter_snapshot_files(source_root)) if source_root.exists() else 0,
    })
SOURCE_INVENTORY.append({
    "category": "benchmark_workbook",
    "source_root": str(SOURCE_WORKBOOK),
    "exists": SOURCE_WORKBOOK.is_file(),
    "files": 1,
})
SOURCE_INVENTORY.append({
    "category": "notebooks",
    "source_root": " | ".join(map(str, SOURCE_NOTEBOOKS)) or "not found below source root",
    "exists": bool(SOURCE_NOTEBOOKS),
    "files": len(SOURCE_NOTEBOOKS),
})

display(pd.DataFrame(SOURCE_INVENTORY))

,category,source_root,exists,files
0,runs,/content/drive/MyDrive/Unlearning_Project/prelangchain_ab_v1_2_workspace/runs/prelangchain_ab_v1_2_api_hardened,True,52
1,artifacts,/content/drive/MyDrive/Unlearning_Project/prelangchain_ab_v1_2_workspace/artifacts/prelangchain_ab_v1_2_api_hardened,True,7
2,reports,/content/drive/MyDrive/Unlearning_Project/prelangchain_ab_v1_2_workspace/reports/prelangchain_ab_v1_2_api_hardened,True,42
3,configs,/content/drive/MyDrive/Unlearning_Project/prelangchain_ab_v1_2_workspace/configs/prelangchain_ab_v1_2_api_hardened,True,17
4,backups,/content/drive/MyDrive/Unlearning_Project/prelangchain_ab_v1_2_workspace/backups/prelangchain_ab_v1_2_api_hardened,True,3
5,benchmark_workbook,/content/drive/MyDrive/Unlearning_Project/prelangchain_ab_v1_2_workspace/data/Unlearning_Codebook_Local_Context_Test_Set.xlsx,True,1
6,notebooks,not found below source root,False,0


In [6]:
SNAPSHOT_LOCK_PATH = SNAPSHOT_ROOT / "SNAPSHOT_LOCK.json"
SNAPSHOT_MANIFEST_PATH = SNAPSHOT_ROOT / "snapshot_manifest.csv"
SNAPSHOT_METADATA_PATH = SNAPSHOT_ROOT / "snapshot_metadata.json"


def create_snapshot() -> pd.DataFrame:
    if SNAPSHOT_LOCK_PATH.exists():
        print("Locked snapshot already exists; verifying without modification.")
        if not SNAPSHOT_MANIFEST_PATH.is_file() or not SNAPSHOT_METADATA_PATH.is_file():
            raise FileNotFoundError(
                "Snapshot lock exists but manifest or metadata is missing. "
                "Do not modify the frozen directory manually."
            )
        lock = json.loads(SNAPSHOT_LOCK_PATH.read_text(encoding="utf-8"))
        observed_manifest_hash = sha256_file(SNAPSHOT_MANIFEST_PATH)
        observed_metadata_hash = sha256_file(SNAPSHOT_METADATA_PATH)
        if observed_manifest_hash != lock.get("manifest_sha256"):
            raise RuntimeError("Frozen snapshot manifest no longer matches its lock hash.")
        if observed_metadata_hash != lock.get("snapshot_metadata_sha256"):
            raise RuntimeError("Frozen snapshot metadata no longer matches its lock hash.")
        return pd.read_csv(SNAPSHOT_MANIFEST_PATH)

    SNAPSHOT_SOURCE_ROOT.mkdir(parents=True, exist_ok=True)
    records: list[dict[str, Any]] = []

    for category, source_root in SOURCE_TREES.items():
        if not source_root.exists():
            records.append({
                "category": category,
                "status": "source_missing",
                "source": str(source_root),
                "snapshot_relative_path": None,
                "size_bytes": None,
                "sha256": None,
            })
            continue
        for source_file in iter_snapshot_files(source_root):
            relative = source_file.relative_to(source_root)
            destination = SNAPSHOT_SOURCE_ROOT / category / SOURCE_PROTOCOL / relative
            record = verified_atomic_copy(source_file, destination)
            record.update({
                "category": category,
                "snapshot_relative_path": str(destination.relative_to(SNAPSHOT_ROOT)),
            })
            records.append(record)

    workbook_destination = SNAPSHOT_SOURCE_ROOT / "data" / SOURCE_WORKBOOK.name
    workbook_record = verified_atomic_copy(SOURCE_WORKBOOK, workbook_destination)
    workbook_record.update({
        "category": "benchmark_workbook",
        "snapshot_relative_path": str(workbook_destination.relative_to(SNAPSHOT_ROOT)),
    })
    records.append(workbook_record)

    for notebook_path in SOURCE_NOTEBOOKS:
        notebook_destination = SNAPSHOT_SOURCE_ROOT / "notebooks" / notebook_path.name
        notebook_record = verified_atomic_copy(notebook_path, notebook_destination)
        notebook_record.update({
            "category": "notebook",
            "snapshot_relative_path": str(notebook_destination.relative_to(SNAPSHOT_ROOT)),
        })
        records.append(notebook_record)

    manifest = pd.DataFrame(records)
    atomic_write_dataframe(manifest, SNAPSHOT_MANIFEST_PATH)
    manifest_digest = sha256_file(SNAPSHOT_MANIFEST_PATH)

    metadata = {
        "snapshot_name": SNAPSHOT_NAME,
        "created_at_utc": utc_now_iso(),
        "source_protocol": SOURCE_PROTOCOL,
        "source_project_root": str(SOURCE_PROJECT_ROOT),
        "snapshot_root": str(SNAPSHOT_ROOT),
        "source_workbook": str(SOURCE_WORKBOOK),
        "source_notebooks": [str(path) for path in SOURCE_NOTEBOOKS],
        "manifest_sha256": manifest_digest,
        "file_records": len(manifest),
        "note": "Immutable source snapshot for post-hoc analysis; raw source directories remain untouched.",
    }
    atomic_write_text(SNAPSHOT_METADATA_PATH, json.dumps(metadata, indent=2))
    atomic_write_text(
        SNAPSHOT_LOCK_PATH,
        json.dumps({
            "locked_at_utc": utc_now_iso(),
            "manifest_sha256": manifest_digest,
            "snapshot_metadata_sha256": sha256_file(SNAPSHOT_METADATA_PATH),
        }, indent=2),
    )
    return manifest


def verify_snapshot(manifest: pd.DataFrame) -> pd.DataFrame:
    verification_rows = []
    for _, record in manifest.iterrows():
        relative = record.get("snapshot_relative_path")
        expected_hash = record.get("sha256")
        if not isinstance(relative, str) or not relative:
            continue
        path = SNAPSHOT_ROOT / relative
        exists = path.is_file()
        observed_hash = sha256_file(path) if exists else None
        verification_rows.append({
            "snapshot_relative_path": relative,
            "exists": exists,
            "expected_sha256": expected_hash,
            "observed_sha256": observed_hash,
            "verified": exists and observed_hash == expected_hash,
        })
    verification = pd.DataFrame(verification_rows)
    if verification.empty or not verification["verified"].all():
        failures = verification[~verification["verified"]] if not verification.empty else verification
        raise RuntimeError(
            "Frozen snapshot verification failed:\n" + failures.to_string(index=False)
        )
    return verification


SNAPSHOT_MANIFEST = create_snapshot()
SNAPSHOT_VERIFICATION = verify_snapshot(SNAPSHOT_MANIFEST)
print("Snapshot verified files:", len(SNAPSHOT_VERIFICATION))
print("Snapshot root:", SNAPSHOT_ROOT)
display(SNAPSHOT_MANIFEST.groupby(["category", "status"], dropna=False).size().rename("files").reset_index())

Snapshot verified files: 122
Snapshot root: /content/drive/MyDrive/Unlearning_Project/prelangchain_ab_v1_2_workspace/frozen_runs/prelangchain_v12_discovery_r1_complete


,category,status,files
0,artifacts,copied,7
1,backups,copied,3
2,benchmark_workbook,copied,1
3,configs,copied,17
4,reports,copied,42
5,runs,copied,52


In [7]:
# Required-artifact gate: the completed source experiment must be recoverable from the snapshot.
SNAPSHOT_RUNS_ROOT = SNAPSHOT_SOURCE_ROOT / "runs" / SOURCE_PROTOCOL
SNAPSHOT_ARTIFACTS_ROOT = SNAPSHOT_SOURCE_ROOT / "artifacts" / SOURCE_PROTOCOL
SNAPSHOT_REPORTS_ROOT = SNAPSHOT_SOURCE_ROOT / "reports" / SOURCE_PROTOCOL
SNAPSHOT_CONFIG_ROOT = SNAPSHOT_SOURCE_ROOT / "configs" / SOURCE_PROTOCOL
SNAPSHOT_BACKUPS_ROOT = SNAPSHOT_SOURCE_ROOT / "backups" / SOURCE_PROTOCOL
SNAPSHOT_DATA_ROOT = SNAPSHOT_SOURCE_ROOT / "data"

required_paths = {
    "benchmark_csv_or_parquet": [
        SNAPSHOT_ARTIFACTS_ROOT / "benchmark_canonical.parquet",
        SNAPSHOT_ARTIFACTS_ROOT / "benchmark_canonical.csv",
    ],
    "phase1_selection": [SNAPSHOT_CONFIG_ROOT / "phase1_selection.json"],
    "phase2_selection": [SNAPSHOT_CONFIG_ROOT / "phase2_selection.json"],
    "phase3_selection": [SNAPSHOT_CONFIG_ROOT / "phase3_selection.json"],
    "final_configuration": [SNAPSHOT_CONFIG_ROOT / "final_configuration.json"],
    "execution_summary": [SNAPSHOT_REPORTS_ROOT / "execution_summary.json"],
}

for phase in ["phase1", "phase2", "phase3", "stability"]:
    for provider in EXPECTED_PROVIDERS:
        required_paths[f"{phase}_{provider}_results"] = [
            SNAPSHOT_RUNS_ROOT / phase / provider / "results.jsonl"
        ]

REQUIRED_ARTIFACT_AUDIT = []
for name, candidates in required_paths.items():
    existing = [path for path in candidates if path.is_file() and path.stat().st_size > 0]
    REQUIRED_ARTIFACT_AUDIT.append({
        "artifact": name,
        "passed": bool(existing),
        "path": str(existing[0]) if existing else " | ".join(map(str, candidates)),
    })

REQUIRED_ARTIFACT_AUDIT = pd.DataFrame(REQUIRED_ARTIFACT_AUDIT)
display(REQUIRED_ARTIFACT_AUDIT)

if STRICT_SOURCE_COMPLETENESS and not REQUIRED_ARTIFACT_AUDIT["passed"].all():
    raise FileNotFoundError(
        "The frozen source run is missing required artifacts:\n"
        + REQUIRED_ARTIFACT_AUDIT[~REQUIRED_ARTIFACT_AUDIT["passed"]].to_string(index=False)
    )

,artifact,passed,path
0,benchmark_csv_or_parquet,True,/content/drive/MyDrive/Unlearning_Project/prelangchain_ab_v1_2_workspace/frozen_runs/prelangchain_v12_discovery_r1_complete/source/artifacts/prelangchain_ab_v1_2_api_hardened/b...
1,phase1_selection,True,/content/drive/MyDrive/Unlearning_Project/prelangchain_ab_v1_2_workspace/frozen_runs/prelangchain_v12_discovery_r1_complete/source/configs/prelangchain_ab_v1_2_api_hardened/pha...
2,phase2_selection,True,/content/drive/MyDrive/Unlearning_Project/prelangchain_ab_v1_2_workspace/frozen_runs/prelangchain_v12_discovery_r1_complete/source/configs/prelangchain_ab_v1_2_api_hardened/pha...
3,phase3_selection,True,/content/drive/MyDrive/Unlearning_Project/prelangchain_ab_v1_2_workspace/frozen_runs/prelangchain_v12_discovery_r1_complete/source/configs/prelangchain_ab_v1_2_api_hardened/pha...
4,final_configuration,True,/content/drive/MyDrive/Unlearning_Project/prelangchain_ab_v1_2_workspace/frozen_runs/prelangchain_v12_discovery_r1_complete/source/configs/prelangchain_ab_v1_2_api_hardened/fin...
5,execution_summary,True,/content/drive/MyDrive/Unlearning_Project/prelangchain_ab_v1_2_workspace/frozen_runs/prelangchain_v12_discovery_r1_complete/source/reports/prelangchain_ab_v1_2_api_hardened/exe...
6,phase1_openai_results,True,/content/drive/MyDrive/Unlearning_Project/prelangchain_ab_v1_2_workspace/frozen_runs/prelangchain_v12_discovery_r1_complete/source/runs/prelangchain_ab_v1_2_api_hardened/phase1...
7,phase1_anthropic_results,True,/content/drive/MyDrive/Unlearning_Project/prelangchain_ab_v1_2_workspace/frozen_runs/prelangchain_v12_discovery_r1_complete/source/runs/prelangchain_ab_v1_2_api_hardened/phase1...
8,phase1_gemini_results,True,/content/drive/MyDrive/Unlearning_Project/prelangchain_ab_v1_2_workspace/frozen_runs/prelangchain_v12_discovery_r1_complete/source/runs/prelangchain_ab_v1_2_api_hardened/phase1...
9,phase2_openai_results,True,/content/drive/MyDrive/Unlearning_Project/prelangchain_ab_v1_2_workspace/frozen_runs/prelangchain_v12_discovery_r1_complete/source/runs/prelangchain_ab_v1_2_api_hardened/phase2...


## 4. Import the frozen source run

All subsequent cells read only from `SNAPSHOT_SOURCE_ROOT`. They do not read the live `runs/`, `configs/`, or `reports/` trees.

In [8]:
def load_json(path: Path, required: bool = True) -> dict[str, Any]:
    if not path.is_file():
        if required:
            raise FileNotFoundError(path)
        return {}
    return json.loads(path.read_text(encoding="utf-8"))


def read_jsonl_latest(path: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Return latest record per run key and a line-level parse audit."""
    latest: dict[str, dict[str, Any]] = {}
    audit_rows = []
    with path.open(encoding="utf-8") as handle:
        lines = handle.readlines()
    for line_number, line in enumerate(lines, start=1):
        if not line.strip():
            continue
        try:
            record = json.loads(line)
            run_key = record.get("run_key")
            if run_key:
                latest[str(run_key)] = record
            audit_rows.append({
                "path": str(path),
                "line_number": line_number,
                "valid_json": True,
                "run_key": run_key,
                "status": record.get("status"),
            })
        except json.JSONDecodeError as exc:
            audit_rows.append({
                "path": str(path),
                "line_number": line_number,
                "valid_json": False,
                "run_key": None,
                "status": None,
                "error": str(exc),
            })
    audit = pd.DataFrame(audit_rows)
    invalid = audit[~audit["valid_json"]] if not audit.empty else audit
    if not invalid.empty:
        raise ValueError("Malformed frozen JSONL:\n" + invalid.to_string(index=False))
    return pd.DataFrame(latest.values()), audit


def parse_json_object(value: Any) -> dict[str, Any]:
    if isinstance(value, dict):
        return value
    if isinstance(value, str) and value.strip():
        try:
            parsed = json.loads(value)
            return parsed if isinstance(parsed, dict) else {}
        except json.JSONDecodeError:
            return {}
    return {}


def flatten_parsed_outputs(frame: pd.DataFrame) -> pd.DataFrame:
    if frame.empty:
        return frame.copy()
    objects = frame.get(
        "parsed_output_json",
        pd.Series([None] * len(frame), index=frame.index),
    ).map(parse_json_object)
    flattened = pd.json_normalize(objects.tolist(), sep="__").add_prefix("out__")
    return pd.concat([frame.reset_index(drop=True), flattened.reset_index(drop=True)], axis=1)


def load_phase_latest(phase: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    frames = []
    audits = []
    for provider in EXPECTED_PROVIDERS:
        path = SNAPSHOT_RUNS_ROOT / phase / provider / "results.jsonl"
        frame, audit = read_jsonl_latest(path)
        if not frame.empty:
            frame["source_results_path"] = str(path)
            frames.append(frame)
        audits.append(audit)
    combined = pd.concat(frames, ignore_index=True, sort=False) if frames else pd.DataFrame()
    combined_audit = pd.concat(audits, ignore_index=True, sort=False) if audits else pd.DataFrame()
    return combined, combined_audit

In [9]:
def load_frozen_benchmark() -> pd.DataFrame:
    parquet_path = SNAPSHOT_ARTIFACTS_ROOT / "benchmark_canonical.parquet"
    csv_path = SNAPSHOT_ARTIFACTS_ROOT / "benchmark_canonical.csv"
    if parquet_path.is_file():
        try:
            data = pd.read_parquet(parquet_path)
            source = parquet_path
        except (ImportError, ModuleNotFoundError, ValueError) as exc:
            if not csv_path.is_file():
                raise
            print(
                "Parquet reader unavailable; using the equivalent frozen CSV:",
                type(exc).__name__,
            )
            data = pd.read_csv(csv_path)
            source = csv_path
    elif csv_path.is_file():
        data = pd.read_csv(csv_path)
        source = csv_path
    else:
        workbook = SNAPSHOT_DATA_ROOT / "Unlearning_Codebook_Local_Context_Test_Set.xlsx"
        data = pd.read_excel(workbook, sheet_name="GPT Test")
        source = workbook
    data = data.copy()
    data["row_id"] = data["row_id"].astype(str)
    print("Frozen benchmark source:", source)
    return data


BENCHMARK = load_frozen_benchmark()

SOURCE_SELECTIONS = {
    "phase1": load_json(SNAPSHOT_CONFIG_ROOT / "phase1_selection.json"),
    "phase2": load_json(SNAPSHOT_CONFIG_ROOT / "phase2_selection.json"),
    "phase3": load_json(SNAPSHOT_CONFIG_ROOT / "phase3_selection.json"),
    "final": load_json(SNAPSHOT_CONFIG_ROOT / "final_configuration.json"),
}
SOURCE_EXECUTION_SUMMARY = load_json(SNAPSHOT_REPORTS_ROOT / "execution_summary.json")

SOURCE_DEFINITIONS = pd.DataFrame()
definition_registry_path = SNAPSHOT_CONFIG_ROOT / "definition_registry.csv"
if definition_registry_path.is_file():
    SOURCE_DEFINITIONS = pd.read_csv(definition_registry_path)

SOURCE_MODELS = pd.DataFrame()
model_protocol_path = SNAPSHOT_CONFIG_ROOT / "model_protocol.csv"
if model_protocol_path.is_file():
    SOURCE_MODELS = pd.read_csv(model_protocol_path)

PHASE_RAW: dict[str, pd.DataFrame] = {}
JSONL_AUDITS = []
for phase in ["phase1", "phase2", "phase3", "stability"]:
    PHASE_RAW[phase], audit = load_phase_latest(phase)
    JSONL_AUDITS.append(audit.assign(phase=phase))

JSONL_AUDIT = pd.concat(JSONL_AUDITS, ignore_index=True, sort=False)

IMPORT_SUMMARY = pd.DataFrame([
    {
        "phase": phase,
        "latest_run_keys": len(frame),
        "successful": int(frame.get("status", pd.Series(dtype=str)).eq("ok").sum()),
        "errors": int(frame.get("status", pd.Series(dtype=str)).eq("error").sum()),
        "providers": int(frame.get("provider", pd.Series(dtype=str)).nunique()),
    }
    for phase, frame in PHASE_RAW.items()
])

display(IMPORT_SUMMARY)
print("Source selections:")
print(json.dumps(SOURCE_SELECTIONS, indent=2, default=str))

Frozen benchmark source: /content/drive/MyDrive/Unlearning_Project/prelangchain_ab_v1_2_workspace/frozen_runs/prelangchain_v12_discovery_r1_complete/source/artifacts/prelangchain_ab_v1_2_api_hardened/benchmark_canonical.parquet


,phase,latest_run_keys,successful,errors,providers
0,phase1,882,882,0,3
1,phase2,756,756,0,3
2,phase3,292,292,0,3
3,stability,630,630,0,3


Source selections:
{
  "phase1": {
    "selected_condition_ids": [
      "P1_DIRECT",
      "P3_D2_CURRENT"
    ],
    "selection_basis": "preregistered constrained ranking against common-final gold",
    "provisional_gold": true,
    "created_at_utc": "2026-07-22T21:40:58.573684+00:00",
    "phase1_ranking_sha256": "52f8a5ecf0e60ed480f97c1f9bbf06612247be7051f39555879f2fb46bf3f107"
  },
  "phase2": {
    "selected_condition_id": "P1_DIRECT__C1_target",
    "selection_basis": "preregistered constrained ranking",
    "provisional_gold": true,
    "phase2_ranking_sha256": "78d7c4e7b646dd244df6d5fd00c183957c6d8eedbc964980fb8f572cc8c8f6cf",
    "phase2_progress": [
      {
        "phase": "phase2",
        "provider": "openai",
        "expected_tasks": 252,
        "successful_tasks": 252,
        "remaining_tasks": 0,
        "latest_error_keys": 0,
        "complete": true,
        "results_path": "/content/drive/MyDrive/Unlearning_Project/prelangchain_ab_v1_2_workspace/runs/prelangchai

In [10]:
# Save imported latest-record snapshots as compact analysis inputs.
IMPORTED_ROOT = ANALYSIS_ROOT / "imported"
IMPORTED_ROOT.mkdir(parents=True, exist_ok=True)

for phase, frame in PHASE_RAW.items():
    frame.to_csv(
        IMPORTED_ROOT / f"{phase}_latest_records.csv.gz",
        index=False,
        compression="gzip",
    )
BENCHMARK.to_csv(IMPORTED_ROOT / "benchmark_frozen.csv", index=False)
IMPORT_SUMMARY.to_csv(IMPORTED_ROOT / "import_summary.csv", index=False)
JSONL_AUDIT.to_csv(IMPORTED_ROOT / "jsonl_parse_audit.csv", index=False)

print("Imported copies saved under:", IMPORTED_ROOT)

Imported copies saved under: /content/drive/MyDrive/Unlearning_Project/prelangchain_ab_v1_2_workspace/analysis/prelangchain_posthoc_v1_3_1/imported


## 5. Separate prompt-input and gold-label provenance hashes

In [11]:
def normalize_space(value: Any) -> str:
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return ""
    return re.sub(r"\s+", " ", str(value)).strip()


def normalize_for_match(value: Any) -> str:
    text = unicodedata.normalize("NFKC", normalize_space(value))
    translation = str.maketrans({
        "“": '"', "”": '"', "‘": "'", "’": "'",
        "–": "-", "—": "-", " ": " ",
    })
    return normalize_space(text.translate(translation)).casefold()


def dataframe_sha256(frame: pd.DataFrame, columns: Sequence[str]) -> str:
    available = [column for column in columns if column in frame.columns]
    if not available:
        raise ValueError("No requested columns are available for dataframe hashing.")
    canonical = (
        frame[available]
        .fillna("")
        .astype(str)
        .sort_values(available[0], kind="stable")
        .to_csv(index=False, lineterminator="\n")
    )
    return sha256_text(canonical)


PROMPT_INPUT_HASH_COLUMNS = [
    "row_id",
    "target_text",
    "document_title",
    "pdf_page",
    "section_heading",
    "paragraph_order",
    "context_previous_2",
    "context_previous_1",
    "context_target",
    "context_next_1",
    "context_next_2",
]

GOLD_HASH_COLUMNS = [
    "row_id",
    "gold_historical",
    "gold_old_definition",
    "gold_current_definition",
    "gold_final_definition",
    "gold_target",
    "gold_agency",
]

# The prompt-input hash is immutable for all post-hoc rescoring. The source-gold
# hash captures the labels present at the time of the paid experiment. A second
# analysis-gold hash is recomputed after an adjudication workbook is imported.
PROMPT_INPUT_SHA256 = dataframe_sha256(BENCHMARK, PROMPT_INPUT_HASH_COLUMNS)
SOURCE_GOLD_LABELS_SHA256 = dataframe_sha256(BENCHMARK, GOLD_HASH_COLUMNS)
ANALYSIS_GOLD_LABELS_SHA256 = SOURCE_GOLD_LABELS_SHA256
GOLD_LABELS_SHA256 = ANALYSIS_GOLD_LABELS_SHA256
FULL_BENCHMARK_SHA256 = dataframe_sha256(
    BENCHMARK,
    list(dict.fromkeys(PROMPT_INPUT_HASH_COLUMNS + GOLD_HASH_COLUMNS)),
)

HASH_MANIFEST = {
    "created_at_utc": utc_now_iso(),
    "analysis_version": ANALYSIS_VERSION,
    "frozen_source_protocol": SOURCE_PROTOCOL,
    "prompt_input_sha256": PROMPT_INPUT_SHA256,
    "source_gold_labels_sha256": SOURCE_GOLD_LABELS_SHA256,
    "analysis_gold_labels_sha256": ANALYSIS_GOLD_LABELS_SHA256,
    "full_analysis_benchmark_sha256": FULL_BENCHMARK_SHA256,
    "prompt_input_columns": PROMPT_INPUT_HASH_COLUMNS,
    "gold_label_columns": GOLD_HASH_COLUMNS,
    "source_dataset_sha256": SOURCE_EXECUTION_SUMMARY.get("dataset_sha256"),
    "adjudication_workbook_sha256": None,
    "run_key_policy": (
        "Completed v1.2 run keys remain untouched and continue to identify the "
        "original prompt inputs. Human-label edits change only the analysis-gold "
        "hash and never make a previous API request appear missing."
    ),
}

atomic_write_text(
    MANIFEST_ROOT / "analysis_dataset_hashes.json",
    json.dumps(HASH_MANIFEST, indent=2),
)
print(json.dumps(HASH_MANIFEST, indent=2))

{
  "created_at_utc": "2026-07-24T00:45:03.817158+00:00",
  "analysis_version": "prelangchain_posthoc_v1_3_1",
  "frozen_source_protocol": "prelangchain_ab_v1_2_api_hardened",
  "prompt_input_sha256": "107cdf61cc742a491c993b1f1eb27d69c108ce9ef359379664eba471270170af",
  "source_gold_labels_sha256": "d471754373ae679bd04b83a218e1032dc440d63f142d9dd9f187f284993cc1fb",
  "analysis_gold_labels_sha256": "d471754373ae679bd04b83a218e1032dc440d63f142d9dd9f187f284993cc1fb",
  "full_analysis_benchmark_sha256": "ab203fea011cbf56eb75efcb5a2448c713b7640dc8f1deedfdfe2e9673f34ea0",
  "prompt_input_columns": [
    "row_id",
    "target_text",
    "document_title",
    "pdf_page",
    "section_heading",
    "paragraph_order",
    "context_previous_2",
    "context_previous_1",
    "context_target",
    "context_next_1",
    "context_next_2"
  ],
  "gold_label_columns": [
    "row_id",
    "gold_historical",
    "gold_old_definition",
    "gold_current_definition",
    "gold_final_definition",
    "gold_

## 6. Exact document identity and EPA correction

The earlier analysis used `str.contains("EPA")`, which also matched the substring **epa** inside “Preparedness.” This notebook assigns a permanent document ID and uses exact normalized equality for the EPA Katrina lessons document.

In [12]:
def slugify(value: Any) -> str:
    text = normalize_for_match(value)
    text = re.sub(r"[^a-z0-9]+", "_", text).strip("_")
    return text[:80] or "unknown_document"


EPA_TITLE_NORMALIZED = normalize_for_match(EPA_DOCUMENT_TITLE)
unique_titles = sorted(BENCHMARK["document_title"].dropna().astype(str).unique())
exact_epa_titles = [
    title for title in unique_titles
    if normalize_for_match(title) == EPA_TITLE_NORMALIZED
]

if len(exact_epa_titles) != 1:
    display(pd.DataFrame({"available_document_title": unique_titles}))
    raise ValueError(
        "EPA_DOCUMENT_TITLE must exactly identify one document after Unicode/whitespace "
        f"normalization. Found {len(exact_epa_titles)} matches."
    )

EPA_CANONICAL_TITLE = exact_epa_titles[0]

DOCUMENT_REGISTRY = pd.DataFrame({"document_title": unique_titles})
DOCUMENT_REGISTRY["document_id"] = DOCUMENT_REGISTRY["document_title"].map(slugify)
DOCUMENT_REGISTRY.loc[
    DOCUMENT_REGISTRY["document_title"].eq(EPA_CANONICAL_TITLE),
    "document_id",
] = "epa_katrina_lessons"
DOCUMENT_REGISTRY["is_epa"] = DOCUMENT_REGISTRY["document_id"].eq("epa_katrina_lessons")

BENCHMARK = BENCHMARK.merge(
    DOCUMENT_REGISTRY,
    on="document_title",
    how="left",
    validate="many_to_one",
)

NAIVE_EPA_MATCH = BENCHMARK["document_title"].str.contains("EPA", case=False, na=False)
EXACT_EPA_MATCH = BENCHMARK["document_id"].eq("epa_katrina_lessons")

EPA_MATCH_AUDIT = BENCHMARK.loc[
    NAIVE_EPA_MATCH | EXACT_EPA_MATCH,
    ["row_id", "document_id", "document_title"],
].copy()
EPA_MATCH_AUDIT["naive_contains_epa"] = NAIVE_EPA_MATCH[NAIVE_EPA_MATCH | EXACT_EPA_MATCH].to_numpy()
EPA_MATCH_AUDIT["exact_epa_identity"] = EXACT_EPA_MATCH[NAIVE_EPA_MATCH | EXACT_EPA_MATCH].to_numpy()
EPA_MATCH_AUDIT["false_positive_from_substring"] = (
    EPA_MATCH_AUDIT["naive_contains_epa"]
    & ~EPA_MATCH_AUDIT["exact_epa_identity"]
)

print("Exact EPA document:", EPA_CANONICAL_TITLE)
print("Exact EPA rows:", int(EXACT_EPA_MATCH.sum()))
display(DOCUMENT_REGISTRY)
display(EPA_MATCH_AUDIT)

Exact EPA document: Lessons Learned: EPA’s Response to Hurricane Katrina
Exact EPA rows: 21


,document_title,document_id,is_epa
0,"Hurricane Katrina: GAO’s Preliminary Observations Regarding Preparedness, Response, and Recovery",hurricane_katrina_gao_s_preliminary_observations_regarding_preparedness_response,False
1,Lessons Learned: EPA’s Response to Hurricane Katrina,epa_katrina_lessons,True
2,The Katrina Effect on American Preparedness — A report on the lessons Americans learned in watching the Katrina catastrophe unfold,the_katrina_effect_on_american_preparedness_a_report_on_the_lessons_americans_le,False


,row_id,document_id,document_title,naive_contains_epa,exact_epa_identity,false_positive_from_substring
0,P001,epa_katrina_lessons,Lessons Learned: EPA’s Response to Hurricane Katrina,True,True,False
1,P002,epa_katrina_lessons,Lessons Learned: EPA’s Response to Hurricane Katrina,True,True,False
2,P003,epa_katrina_lessons,Lessons Learned: EPA’s Response to Hurricane Katrina,True,True,False
3,P004,epa_katrina_lessons,Lessons Learned: EPA’s Response to Hurricane Katrina,True,True,False
4,P005,epa_katrina_lessons,Lessons Learned: EPA’s Response to Hurricane Katrina,True,True,False
5,P006,epa_katrina_lessons,Lessons Learned: EPA’s Response to Hurricane Katrina,True,True,False
6,P007,epa_katrina_lessons,Lessons Learned: EPA’s Response to Hurricane Katrina,True,True,False
7,P008,epa_katrina_lessons,Lessons Learned: EPA’s Response to Hurricane Katrina,True,True,False
8,P009,epa_katrina_lessons,Lessons Learned: EPA’s Response to Hurricane Katrina,True,True,False
9,P010,epa_katrina_lessons,Lessons Learned: EPA’s Response to Hurricane Katrina,True,True,False


## 7. Parse provider outputs and reconstruct every workflow

In [13]:
def successful_latest(frame: pd.DataFrame) -> pd.DataFrame:
    if frame.empty:
        return frame.copy()
    data = frame[frame["status"].eq("ok")].copy()
    if "completed_at_utc" in data.columns:
        data = data.sort_values("completed_at_utc")
    data = data.drop_duplicates("run_key", keep="last")
    return flatten_parsed_outputs(data)


PHASE_SUCCESS = {
    phase: successful_latest(frame)
    for phase, frame in PHASE_RAW.items()
}

PHASE1 = PHASE_SUCCESS["phase1"].copy()
PHASE2_NEW = PHASE_SUCCESS["phase2"].copy()

phase1_selected_ids = list(
    SOURCE_SELECTIONS["phase1"].get("selected_condition_ids", [])
)
if not phase1_selected_ids:
    raise RuntimeError("The frozen Phase 1 selection contains no selected condition IDs.")

PHASE2_C1_REUSED = PHASE1[
    PHASE1["condition_id"].isin(phase1_selected_ids)
].copy()
PHASE2_C1_REUSED["source_phase"] = "phase1_reused"
PHASE2_C1_REUSED["phase"] = "phase2"
PHASE2_C1_REUSED["condition_id"] = (
    PHASE2_C1_REUSED["condition_id"].astype(str) + "__C1_target"
)
PHASE2_C1_REUSED["context_id"] = "C1_target"

if not PHASE2_NEW.empty:
    PHASE2_NEW["source_phase"] = "phase2_new_call"

PHASE2 = pd.concat(
    [PHASE2_C1_REUSED, PHASE2_NEW],
    ignore_index=True,
    sort=False,
)

print({
    "phase1_successful": len(PHASE1),
    "phase2_new_successful": len(PHASE2_NEW),
    "phase2_c1_reused": len(PHASE2_C1_REUSED),
    "phase2_unified": len(PHASE2),
})

{'phase1_successful': 882, 'phase2_new_successful': 756, 'phase2_c1_reused': 252, 'phase2_unified': 1008}


In [14]:
MERGE_KEYS = ["provider", "condition_id", "row_id", "seed", "repeat_id"]
USAGE_COLUMNS = [
    "input_tokens",
    "output_tokens",
    "total_tokens",
    "reasoning_tokens",
    "cached_input_tokens",
    "cache_creation_input_tokens",
    "estimated_cost_usd",
    "latency_seconds",
]


def assemble_workflow_predictions(raw_success: pd.DataFrame) -> pd.DataFrame:
    if raw_success.empty:
        return raw_success.copy()
    joint = raw_success[raw_success["stage"].eq("joint")].copy()
    binary = raw_success[raw_success["stage"].eq("binary")].copy()
    target = raw_success[raw_success["stage"].eq("target")].copy()

    if not joint.empty:
        joint["workflow_complete"] = True
        joint["stage2_called"] = False
        joint["source_stage"] = "joint"

    if not binary.empty:
        target_fields = MERGE_KEYS + [
            "out__target_type",
            "out__agency",
            "out__target_evidence_quote",
            "out__target_evidence_scope",
            "out__confidence",
            "out__needs_human_review",
            "parsed_output_json",
            "source_blocks_json",
            "schema_id",
        ] + [column for column in USAGE_COLUMNS if column in target.columns]
        target_keep = target[[column for column in target_fields if column in target.columns]].copy()
        rename = {
            "out__target_type": "stage2__target_type",
            "out__agency": "stage2__agency",
            "out__target_evidence_quote": "stage2__target_evidence_quote",
            "out__target_evidence_scope": "stage2__target_evidence_scope",
            "out__confidence": "stage2__confidence",
            "out__needs_human_review": "stage2__needs_human_review",
            "parsed_output_json": "stage2__parsed_output_json",
            "source_blocks_json": "stage2__source_blocks_json",
            "schema_id": "stage2__schema_id",
            **{
                column: f"stage2__{column}"
                for column in USAGE_COLUMNS
                if column in target_keep.columns
            },
        }
        target_keep = target_keep.rename(columns=rename)
        binary = binary.merge(target_keep, on=MERGE_KEYS, how="left", validate="one_to_one")
        positive = binary["out__unlearning_present"].eq(True)
        binary["out__target_type"] = np.where(
            positive,
            binary.get("stage2__target_type"),
            "none",
        )
        binary["out__agency"] = np.where(
            positive,
            binary.get("stage2__agency"),
            None,
        )
        binary["stage2_called"] = (
            positive
            & binary.get(
                "stage2__target_type",
                pd.Series(index=binary.index, dtype=object),
            ).notna()
        )
        binary["workflow_complete"] = (~positive) | binary["stage2_called"]
        binary["source_stage"] = "binary_plus_conditional_target"
        for column in USAGE_COLUMNS:
            left = pd.to_numeric(binary.get(column), errors="coerce").fillna(0)
            right = pd.to_numeric(binary.get(f"stage2__{column}"), errors="coerce").fillna(0)
            binary[column] = left + right

    frames = [frame for frame in [joint, binary] if not frame.empty]
    return pd.concat(frames, ignore_index=True, sort=False) if frames else pd.DataFrame()


PHASE3 = assemble_workflow_predictions(PHASE_SUCCESS["phase3"])
STABILITY = assemble_workflow_predictions(PHASE_SUCCESS["stability"])

WORKFLOW_IMPORT_SUMMARY = pd.DataFrame([
    {"dataset": "phase1", "rows": len(PHASE1)},
    {"dataset": "phase2_unified", "rows": len(PHASE2)},
    {"dataset": "phase3_unified", "rows": len(PHASE3)},
    {"dataset": "stability_unified", "rows": len(STABILITY)},
])
display(WORKFLOW_IMPORT_SUMMARY)

,dataset,rows
0,phase1,882
1,phase2_unified,1008
2,phase3_unified,252
3,stability_unified,630


## 8. Corrected, scope-aware evidence audit

The audit reports two quote standards:

- **strict:** the whitespace-collapsed quote occurs verbatim in the claimed block;
- **typography-normalized:** Unicode quotation marks, dashes, case, and PDF whitespace are normalized before matching.

A quote found in another supplied block but not in its claimed scope is reported as a scope error, not silently accepted. Token-only similarity is diagnostic and never makes evidence valid.

In [15]:
SCHEMA_REQUIRED_KEYS = {
    "simple_joint": {
        "unlearning_present", "unlearning_mode", "change_type", "evidence",
        "target_type", "agency", "confidence", "needs_human_review",
    },
    "checklist_joint": {
        "prior_state", "inadequacy_or_failure", "departure_or_reconfiguration",
        "all_required_elements_present", "missing_elements", "unlearning_present",
        "unlearning_mode", "change_type", "target_type", "agency", "confidence",
        "needs_human_review",
    },
    "simple_binary": {
        "unlearning_present", "unlearning_mode", "change_type", "evidence",
        "confidence", "needs_human_review",
    },
    "checklist_binary": {
        "prior_state", "inadequacy_or_failure", "departure_or_reconfiguration",
        "all_required_elements_present", "missing_elements", "unlearning_present",
        "unlearning_mode", "change_type", "confidence", "needs_human_review",
    },
    "stage2_target": {
        "target_type", "agency", "target_evidence_quote",
        "target_evidence_scope", "confidence", "needs_human_review",
    },
}


def parse_source_blocks(value: Any) -> dict[str, str]:
    parsed = parse_json_object(value)
    return {str(key): normalize_space(block) for key, block in parsed.items()}


def extract_evidence_items(parsed: dict[str, Any]) -> list[dict[str, Any]]:
    if isinstance(parsed.get("evidence"), list):
        return [dict(item) for item in parsed["evidence"] if isinstance(item, dict)]
    items = []
    for key in ["prior_state", "inadequacy_or_failure", "departure_or_reconfiguration"]:
        element = parsed.get(key)
        if isinstance(element, dict):
            items.append({
                "element": key,
                "quote": element.get("quote"),
                "source_scope": element.get("source_scope"),
                "identified": element.get("identified"),
            })
    if "target_evidence_quote" in parsed:
        items.append({
            "element": "departure_or_reconfiguration",
            "quote": parsed.get("target_evidence_quote"),
            "source_scope": parsed.get("target_evidence_scope"),
            "identified": bool(parsed.get("target_evidence_quote")),
        })
    return items


def token_sequence(value: Any) -> str:
    return " ".join(re.findall(r"[a-z0-9]+", normalize_for_match(value)))


def quote_match_details(
    quote: Any,
    claimed_scope: Any,
    blocks: dict[str, str],
) -> dict[str, Any]:
    quote_raw = normalize_space(quote)
    scope = normalize_space(claimed_scope)
    if not quote_raw:
        absent_valid = scope == "absent"
        return {
            "strict_match": absent_valid,
            "normalized_match": absent_valid,
            "token_sequence_match": absent_valid,
            "claimed_scope_valid": absent_valid,
            "found_in_scopes": [],
            "format_only_match": False,
            "wrong_scope": False,
        }

    strict_scopes = [
        key for key, source in blocks.items()
        if quote_raw in normalize_space(source)
    ]
    normalized_quote = normalize_for_match(quote_raw)
    normalized_scopes = [
        key for key, source in blocks.items()
        if normalized_quote and normalized_quote in normalize_for_match(source)
    ]
    token_quote = token_sequence(quote_raw)
    token_scopes = [
        key for key, source in blocks.items()
        if token_quote and token_quote in token_sequence(source)
    ]

    strict_match = scope in strict_scopes
    normalized_match = scope in normalized_scopes
    token_match = scope in token_scopes
    found_in_scopes = sorted(set(strict_scopes + normalized_scopes + token_scopes))
    return {
        "strict_match": strict_match,
        "normalized_match": normalized_match,
        "token_sequence_match": token_match,
        "claimed_scope_valid": normalized_match,
        "found_in_scopes": found_in_scopes,
        "format_only_match": (not strict_match) and normalized_match,
        "wrong_scope": bool(found_in_scopes) and scope not in found_in_scopes,
    }

In [16]:
def audit_one_prediction(record: pd.Series) -> tuple[dict[str, Any], list[dict[str, Any]]]:
    parsed = parse_json_object(record.get("parsed_output_json"))
    schema_id = normalize_space(record.get("schema_id"))
    blocks = parse_source_blocks(record.get("source_blocks_json"))

    required = SCHEMA_REQUIRED_KEYS.get(schema_id, set())
    missing_schema_keys = sorted(required - set(parsed))
    schema_valid_structural = bool(parsed) and not missing_schema_keys

    items = extract_evidence_items(parsed)
    quote_rows: list[dict[str, Any]] = []
    for item_index, item in enumerate(items):
        details = quote_match_details(
            item.get("quote"),
            item.get("source_scope"),
            blocks,
        )
        quote_rows.append({
            "run_key": record.get("run_key"),
            "phase": record.get("phase"),
            "provider": record.get("provider"),
            "condition_id": record.get("condition_id"),
            "row_id": str(record.get("row_id")),
            "seed": record.get("seed"),
            "item_index": item_index,
            "element": item.get("element"),
            "identified": item.get("identified"),
            "quote": item.get("quote"),
            "claimed_scope": item.get("source_scope"),
            **details,
        })

    all_quotes_strict = all(row["strict_match"] for row in quote_rows) if quote_rows else True
    all_quotes_normalized = all(row["normalized_match"] for row in quote_rows) if quote_rows else True

    positive = bool(
        parsed.get(
            "unlearning_present",
            True if schema_id == "stage2_target" else False,
        )
    )

    departure_rows = [
        row for row in quote_rows
        if row.get("element") == "departure_or_reconfiguration"
    ]
    target_departure_strict = any(
        row["strict_match"]
        and row.get("claimed_scope") == "target"
        and normalize_space(row.get("quote"))
        for row in departure_rows
    )
    target_departure_normalized = any(
        row["normalized_match"]
        and row.get("claimed_scope") == "target"
        and normalize_space(row.get("quote"))
        for row in departure_rows
    )

    joint_target_null_rule_valid = True
    if schema_id in {"simple_joint", "checklist_joint"}:
        if positive:
            joint_target_null_rule_valid = parsed.get("target_type") not in {None, "none"}
        else:
            joint_target_null_rule_valid = (
                parsed.get("target_type") == "none"
                and parsed.get("agency") is None
            )

    d3_additive_rule_valid = not (
        record.get("definition_id") == "D3_provisional_adaptive"
        and positive
        and parsed.get("change_type") == "add_capacity_only"
    )

    checklist_consistent = True
    if schema_id in {"checklist_joint", "checklist_binary"}:
        required_elements = [
            "prior_state", "inadequacy_or_failure", "departure_or_reconfiguration"
        ]
        identified = {
            name: bool(parsed.get(name, {}).get("identified"))
            for name in required_elements
        }
        expected_missing = sorted(name for name, value in identified.items() if not value)
        checklist_consistent = (
            sorted(parsed.get("missing_elements", [])) == expected_missing
            and bool(parsed.get("all_required_elements_present")) == all(identified.values())
        )

    evidence_valid_strict = all([
        schema_valid_structural,
        all_quotes_strict,
        (not positive or target_departure_strict),
        joint_target_null_rule_valid,
        d3_additive_rule_valid,
        checklist_consistent,
    ])
    evidence_valid_normalized = all([
        schema_valid_structural,
        all_quotes_normalized,
        (not positive or target_departure_normalized),
        joint_target_null_rule_valid,
        d3_additive_rule_valid,
        checklist_consistent,
    ])

    failure_reasons = []
    if not schema_valid_structural:
        failure_reasons.append("structural_schema")
    if not all_quotes_normalized:
        if any(row["wrong_scope"] for row in quote_rows):
            failure_reasons.append("wrong_source_scope")
        if any(not row["found_in_scopes"] for row in quote_rows):
            failure_reasons.append("quote_not_found")
        if not any(row["wrong_scope"] for row in quote_rows) and not any(
            not row["found_in_scopes"] for row in quote_rows
        ):
            failure_reasons.append("quote_mismatch")
    if positive and not target_departure_normalized:
        failure_reasons.append("no_decisive_target_evidence")
    if not joint_target_null_rule_valid:
        failure_reasons.append("joint_target_null_rule")
    if not d3_additive_rule_valid:
        failure_reasons.append("d3_additive_only")
    if not checklist_consistent:
        failure_reasons.append("checklist_inconsistency")

    summary = {
        "schema_valid_structural": schema_valid_structural,
        "missing_schema_keys": " | ".join(missing_schema_keys),
        "evidence_quote_count": len(quote_rows),
        "all_quotes_strict": all_quotes_strict,
        "all_quotes_normalized": all_quotes_normalized,
        "format_only_quote_count": sum(row["format_only_match"] for row in quote_rows),
        "wrong_scope_quote_count": sum(row["wrong_scope"] for row in quote_rows),
        "unmatched_quote_count": sum(not row["found_in_scopes"] for row in quote_rows),
        "target_departure_strict": target_departure_strict,
        "target_departure_normalized": target_departure_normalized,
        "joint_target_null_rule_valid": joint_target_null_rule_valid,
        "d3_additive_rule_valid": d3_additive_rule_valid,
        "checklist_consistent": checklist_consistent,
        "evidence_valid_strict": evidence_valid_strict,
        "prediction_positive": positive,
        "evidence_valid_normalized": evidence_valid_normalized,
        "evidence_failure_reasons": " | ".join(sorted(set(failure_reasons))),
    }
    return summary, quote_rows


def add_corrected_evidence_audit(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    if frame.empty:
        return frame.copy(), pd.DataFrame()
    summaries = []
    quote_rows = []
    for _, record in frame.iterrows():
        summary, quotes = audit_one_prediction(record)
        summaries.append(summary)
        quote_rows.extend(quotes)
    audited = pd.concat(
        [frame.reset_index(drop=True), pd.DataFrame(summaries)],
        axis=1,
    )
    return audited, pd.DataFrame(quote_rows)


PHASE_AUDITED: dict[str, pd.DataFrame] = {}
EVIDENCE_QUOTE_TABLES = []
for name, frame in {
    "phase1": PHASE1,
    "phase2": PHASE2,
    "phase3": PHASE3,
    "stability": STABILITY,
}.items():
    PHASE_AUDITED[name], quote_table = add_corrected_evidence_audit(frame)
    if not quote_table.empty:
        quote_table["analysis_dataset"] = name
        EVIDENCE_QUOTE_TABLES.append(quote_table)

EVIDENCE_QUOTES = (
    pd.concat(EVIDENCE_QUOTE_TABLES, ignore_index=True, sort=False)
    if EVIDENCE_QUOTE_TABLES
    else pd.DataFrame()
)

EVIDENCE_FAILURE_SUMMARY = pd.concat([
    frame.assign(analysis_dataset=name)
    for name, frame in PHASE_AUDITED.items()
], ignore_index=True, sort=False)

EVIDENCE_FAILURE_SUMMARY["target_departure_failure"] = (
    EVIDENCE_FAILURE_SUMMARY["prediction_positive"].astype(bool)
    & ~EVIDENCE_FAILURE_SUMMARY["target_departure_normalized"].astype(bool)
)

EVIDENCE_FAILURE_SUMMARY = (
    EVIDENCE_FAILURE_SUMMARY
    .groupby(
        ["analysis_dataset", "condition_id", "provider"],
        dropna=False,
    )
    .agg(
        rows=("row_id", "size"),
        schema_valid_rate=("schema_valid_structural", "mean"),
        evidence_valid_strict_rate=("evidence_valid_strict", "mean"),
        evidence_valid_normalized_rate=("evidence_valid_normalized", "mean"),
        format_only_quote_matches=("format_only_quote_count", "sum"),
        wrong_scope_quotes=("wrong_scope_quote_count", "sum"),
        unmatched_quotes=("unmatched_quote_count", "sum"),
        target_departure_failures=("target_departure_failure", "sum"),
    )
    .reset_index()
)

display(EVIDENCE_FAILURE_SUMMARY)

,analysis_dataset,condition_id,provider,rows,schema_valid_rate,evidence_valid_strict_rate,evidence_valid_normalized_rate,format_only_quote_matches,wrong_scope_quotes,unmatched_quotes,target_departure_failures
0,phase1,P1_DIRECT,anthropic,42,1.0,0.880952,0.952381,5,0,2,0
1,phase1,P1_DIRECT,gemini,42,1.0,0.976190,0.976190,0,0,1,1
2,phase1,P1_DIRECT,openai,42,1.0,1.000000,1.000000,0,0,3,0
3,phase1,P2_D1_OLD,anthropic,42,1.0,0.857143,0.928571,7,0,2,1
4,phase1,P2_D1_OLD,gemini,42,1.0,0.928571,0.928571,0,0,4,2
5,phase1,P2_D1_OLD,openai,42,1.0,0.952381,0.952381,0,0,8,1
6,phase1,P2_D2_CURRENT,anthropic,42,1.0,0.880952,0.928571,5,0,2,0
7,phase1,P2_D2_CURRENT,gemini,42,1.0,0.904762,0.904762,0,0,7,1
8,phase1,P2_D2_CURRENT,openai,42,1.0,0.928571,0.928571,0,0,33,1
9,phase1,P2_D3_ADAPTIVE,anthropic,42,1.0,0.857143,0.952381,6,0,1,1


## 9. Import completed adjudication when available

The original `gold_historical` column is immutable. The new workbook may add or update only `gold_old_definition`, `gold_current_definition`, `gold_final_definition`, `gold_target`, `gold_agency`, and adjudication metadata.

In [17]:
def normalize_label(value: Any) -> Optional[str]:
    text = normalize_space(value).casefold()
    if not text:
        return None
    if text in {"yes", "y", "true", "1", "unlearning", "present"} or text.startswith("yes"):
        return "Yes"
    if text in {"no", "n", "false", "0", "not unlearning", "absent"} or text.startswith("no"):
        return "No"
    return None


def label_to_int(value: Any) -> Optional[int]:
    return {"Yes": 1, "No": 0}.get(normalize_label(value))


ADJUDICATION_COLUMNS = [
    "row_id",
    "gold_old_definition",
    "gold_current_definition",
    "gold_final_definition",
    "gold_target",
    "gold_agency",
    "adjudication_status",
    "adjudication_notes",
]

ADJUDICATION_IMPORT_AUDIT = {
    "path": str(ADJUDICATION_COMPLETED_PATH),
    "exists": ADJUDICATION_COMPLETED_PATH.is_file(),
    "rows_imported": 0,
    "benchmark_rows": len(BENCHMARK),
    "historical_labels_changed": False,
    "unknown_row_ids": 0,
    "missing_row_ids": 0,
}

if ADJUDICATION_COMPLETED_PATH.is_file():
    adjudicated = pd.read_excel(ADJUDICATION_COMPLETED_PATH, sheet_name="Adjudication")
    adjudicated["row_id"] = adjudicated["row_id"].astype(str)
    if not adjudicated["row_id"].is_unique:
        raise ValueError("Completed adjudication workbook contains duplicate row_id values.")
    missing_columns = set(ADJUDICATION_COLUMNS) - set(adjudicated.columns)
    if missing_columns:
        raise ValueError(
            "Completed adjudication workbook is missing columns: "
            f"{sorted(missing_columns)}"
        )

    source_ids = set(BENCHMARK["row_id"].astype(str))
    supplied_ids = set(adjudicated["row_id"].astype(str))
    unknown_ids = sorted(supplied_ids - source_ids)
    missing_ids = sorted(source_ids - supplied_ids)
    ADJUDICATION_IMPORT_AUDIT["unknown_row_ids"] = len(unknown_ids)
    ADJUDICATION_IMPORT_AUDIT["missing_row_ids"] = len(missing_ids)
    if unknown_ids:
        raise ValueError(
            "Completed adjudication contains row IDs absent from the frozen benchmark: "
            + ", ".join(unknown_ids[:20])
        )

    if "gold_historical" in adjudicated.columns:
        supplied_historical = adjudicated["gold_historical"].map(normalize_label)
        source_historical = (
            BENCHMARK.set_index("row_id")
            .loc[adjudicated["row_id"], "gold_historical"]
            .map(normalize_label)
            .reset_index(drop=True)
        )
        historical_changed = not supplied_historical.reset_index(drop=True).equals(source_historical)
        ADJUDICATION_IMPORT_AUDIT["historical_labels_changed"] = historical_changed
        if historical_changed:
            raise ValueError(
                "The completed workbook attempts to change immutable historical labels."
            )

    updates = adjudicated[ADJUDICATION_COLUMNS].copy()
    for column in [
        "gold_old_definition", "gold_current_definition", "gold_final_definition"
    ]:
        invalid_raw = updates[column].notna() & updates[column].map(normalize_label).isna()
        if invalid_raw.any():
            raise ValueError(
                f"Invalid Yes/No values in {column}: "
                + ", ".join(map(str, updates.loc[invalid_raw, column].unique()))
            )
        updates[column] = updates[column].map(normalize_label)

    BENCHMARK = BENCHMARK.drop(
        columns=[
            column
            for column in ADJUDICATION_COLUMNS
            if column != "row_id" and column in BENCHMARK.columns
        ]
    ).merge(updates, on="row_id", how="left", validate="one_to_one")
    ADJUDICATION_IMPORT_AUDIT["rows_imported"] = len(updates)

# Recompute only the analysis-label provenance after optional adjudication.
# The prompt-input hash must remain identical to the frozen source run.
ANALYSIS_GOLD_LABELS_SHA256 = dataframe_sha256(BENCHMARK, GOLD_HASH_COLUMNS)
GOLD_LABELS_SHA256 = ANALYSIS_GOLD_LABELS_SHA256
FULL_BENCHMARK_SHA256 = dataframe_sha256(
    BENCHMARK,
    list(dict.fromkeys(PROMPT_INPUT_HASH_COLUMNS + GOLD_HASH_COLUMNS)),
)
assert dataframe_sha256(BENCHMARK, PROMPT_INPUT_HASH_COLUMNS) == PROMPT_INPUT_SHA256

HASH_MANIFEST.update({
    "updated_at_utc": utc_now_iso(),
    "analysis_gold_labels_sha256": ANALYSIS_GOLD_LABELS_SHA256,
    "full_analysis_benchmark_sha256": FULL_BENCHMARK_SHA256,
    "adjudication_workbook_sha256": (
        sha256_file(ADJUDICATION_COMPLETED_PATH)
        if ADJUDICATION_COMPLETED_PATH.is_file()
        else None
    ),
    "adjudication_rows_imported": ADJUDICATION_IMPORT_AUDIT["rows_imported"],
})
atomic_write_text(
    MANIFEST_ROOT / "analysis_dataset_hashes.json",
    json.dumps(HASH_MANIFEST, indent=2),
)
BENCHMARK.to_csv(
    IMPORTED_ROOT / "benchmark_with_analysis_gold.csv",
    index=False,
)

print(json.dumps(ADJUDICATION_IMPORT_AUDIT, indent=2, default=str))
print(json.dumps(HASH_MANIFEST, indent=2, default=str))

{
  "path": "/content/drive/MyDrive/Unlearning_Project/prelangchain_ab_v1_2_workspace/analysis/prelangchain_posthoc_v1_3_1/adjudication/adjudication_completed.xlsx",
  "exists": false,
  "rows_imported": 0,
  "benchmark_rows": 42,
  "historical_labels_changed": false,
  "unknown_row_ids": 0,
  "missing_row_ids": 0
}
{
  "created_at_utc": "2026-07-24T00:45:03.817158+00:00",
  "analysis_version": "prelangchain_posthoc_v1_3_1",
  "frozen_source_protocol": "prelangchain_ab_v1_2_api_hardened",
  "prompt_input_sha256": "107cdf61cc742a491c993b1f1eb27d69c108ce9ef359379664eba471270170af",
  "source_gold_labels_sha256": "d471754373ae679bd04b83a218e1032dc440d63f142d9dd9f187f284993cc1fb",
  "analysis_gold_labels_sha256": "d471754373ae679bd04b83a218e1032dc440d63f142d9dd9f187f284993cc1fb",
  "full_analysis_benchmark_sha256": "ab203fea011cbf56eb75efcb5a2448c713b7640dc8f1deedfdfe2e9673f34ea0",
  "prompt_input_columns": [
    "row_id",
    "target_text",
    "document_title",
    "pdf_page",
    "secti

In [18]:
DEFINITION_GOLD_COLUMN = {
    "D0_none": "gold_historical",
    "D1_old_broad": "gold_old_definition",
    "D2_current_strict": "gold_current_definition",
    "D3_provisional_adaptive": "gold_final_definition",
}


def complete_gold_column(column: str) -> bool:
    return bool(
        column in BENCHMARK.columns
        and BENCHMARK[column].map(normalize_label).notna().all()
    )


def aligned_gold_column(definition_id: str) -> tuple[str, str]:
    preferred = DEFINITION_GOLD_COLUMN.get(definition_id, "gold_final_definition")
    if complete_gold_column(preferred):
        return preferred, "definition-aligned adjudicated gold"
    return "gold_historical", "historical provisional gold"


def common_final_gold_column() -> tuple[str, str]:
    if complete_gold_column("gold_final_definition"):
        return "gold_final_definition", "common final adjudicated gold"
    return "gold_historical", "historical provisional gold"


GOLD_AVAILABILITY = pd.DataFrame([
    {
        "gold_column": column,
        "non_null_rows": int(BENCHMARK.get(column, pd.Series(dtype=object)).map(normalize_label).notna().sum()),
        "complete": complete_gold_column(column),
    }
    for column in [
        "gold_historical",
        "gold_old_definition",
        "gold_current_definition",
        "gold_final_definition",
    ]
])

COMMON_FINAL_COLUMN, COMMON_FINAL_BASIS = common_final_gold_column()
FINAL_GOLD_COMPLETE = complete_gold_column("gold_final_definition")

display(GOLD_AVAILABILITY)
print("Common-final scoring basis:", COMMON_FINAL_BASIS, f"({COMMON_FINAL_COLUMN})")

,gold_column,non_null_rows,complete
0,gold_historical,42,True
1,gold_old_definition,0,False
2,gold_current_definition,0,False
3,gold_final_definition,0,False


Common-final scoring basis: historical provisional gold (gold_historical)


In [19]:
def attach_gold(frame: pd.DataFrame) -> pd.DataFrame:
    if frame.empty:
        return frame.copy()
    benchmark_columns = [
        "row_id",
        "document_id",
        "document_title",
        "is_epa",
        "target_text",
        "gold_historical",
        "gold_old_definition",
        "gold_current_definition",
        "gold_final_definition",
        "gold_target",
        "gold_agency",
    ]
    data = frame.merge(
        BENCHMARK[benchmark_columns],
        on="row_id",
        how="left",
        validate="many_to_one",
    )
    aligned_values = []
    aligned_bases = []
    for _, row in data.iterrows():
        column, basis = aligned_gold_column(str(row.get("definition_id")))
        aligned_values.append(row.get(column))
        aligned_bases.append(f"{basis} ({column})")
    data["gold_definition_aligned"] = aligned_values
    data["gold_definition_aligned_basis"] = aligned_bases
    data["gold_common_final"] = data[COMMON_FINAL_COLUMN]
    data["gold_common_final_basis"] = f"{COMMON_FINAL_BASIS} ({COMMON_FINAL_COLUMN})"
    data["pred_unlearning"] = data.get("out__unlearning_present").map(
        lambda value: "Yes" if value is True else ("No" if value is False else None)
    )
    data["pred_int"] = data["pred_unlearning"].map(label_to_int)
    data["gold_aligned_int"] = data["gold_definition_aligned"].map(label_to_int)
    data["gold_common_int"] = data["gold_common_final"].map(label_to_int)
    data["confidence"] = pd.to_numeric(data.get("out__confidence"), errors="coerce")
    data["probability_yes"] = np.where(
        data["pred_int"].eq(1),
        data["confidence"],
        1 - data["confidence"],
    )
    return data


SCORED = {
    phase: attach_gold(frame)
    for phase, frame in PHASE_AUDITED.items()
}

SCORING_SUMMARY = pd.DataFrame([
    {
        "dataset": phase,
        "rows": len(frame),
        "gold_basis": COMMON_FINAL_BASIS,
        "gold_complete_final": FINAL_GOLD_COMPLETE,
        "rows_scored": int(frame.get("gold_common_int", pd.Series(dtype=float)).notna().sum()),
    }
    for phase, frame in SCORED.items()
])
display(SCORING_SUMMARY)

,dataset,rows,gold_basis,gold_complete_final,rows_scored
0,phase1,882,historical provisional gold,False,882
1,phase2,1008,historical provisional gold,False,1008
2,phase3,252,historical provisional gold,False,252
3,stability,630,historical provisional gold,False,630


## 10. Binary metrics, constraints, Pareto fronts, and honest selection labels

In [20]:
def safe_rate(numerator: float, denominator: float) -> float:
    return float(numerator / denominator) if denominator else np.nan


def binary_metric_record(group: pd.DataFrame, gold_column: str = "gold_common_int") -> dict[str, Any]:
    valid = group.dropna(subset=[gold_column, "pred_int"]).copy()
    if valid.empty:
        return {"n_scored": 0}
    y_true = valid[gold_column].astype(int).to_numpy()
    y_pred = valid["pred_int"].astype(int).to_numpy()
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    probability = pd.to_numeric(valid.get("probability_yes"), errors="coerce")
    brier = float(np.mean((probability - y_true) ** 2)) if probability.notna().all() else np.nan
    return {
        "n_scored": len(valid),
        "human_yes": int(y_true.sum()),
        "human_no": int((1 - y_true).sum()),
        "pred_yes": int(y_pred.sum()),
        "pred_no": int((1 - y_pred).sum()),
        "tp": int(tp),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": (
            (safe_rate(tp, tp + fn) + safe_rate(tn, tn + fp)) / 2
            if (tp + fn) > 0 and (tn + fp) > 0
            else np.nan
        ),
        "precision_yes": precision_score(y_true, y_pred, zero_division=0),
        "recall_yes": recall_score(y_true, y_pred, zero_division=0),
        "specificity": safe_rate(tn, tn + fp),
        "f1_yes": f1_score(y_true, y_pred, zero_division=0),
        "mcc": matthews_corrcoef(y_true, y_pred) if len(set(y_true)) > 1 else np.nan,
        "cohen_kappa": cohen_kappa_score(y_true, y_pred) if len(set(y_true)) > 1 else np.nan,
        "npv": safe_rate(tn, tn + fn),
        "brier_score": brier,
    }


def summarize_binary(
    data: pd.DataFrame,
    group_columns: Sequence[str],
    gold_column: str = "gold_common_int",
) -> pd.DataFrame:
    if data.empty:
        return pd.DataFrame()
    rows = []
    grouper = group_columns[0] if len(group_columns) == 1 else list(group_columns)
    for keys, group in data.groupby(grouper, dropna=False):
        keys = keys if isinstance(keys, tuple) else (keys,)
        row = dict(zip(group_columns, keys))
        row.update(binary_metric_record(group, gold_column))
        row.update({
            "schema_valid_rate": pd.to_numeric(group.get("schema_valid_structural"), errors="coerce").mean(),
            "evidence_valid_strict_rate": pd.to_numeric(group.get("evidence_valid_strict"), errors="coerce").mean(),
            "evidence_valid_rate": pd.to_numeric(group.get("evidence_valid_normalized"), errors="coerce").mean(),
            "mean_confidence": pd.to_numeric(group.get("confidence"), errors="coerce").mean(),
            "input_tokens": pd.to_numeric(group.get("input_tokens"), errors="coerce").sum(min_count=1),
            "output_tokens": pd.to_numeric(group.get("output_tokens"), errors="coerce").sum(min_count=1),
            "estimated_cost_usd": pd.to_numeric(group.get("estimated_cost_usd"), errors="coerce").sum(min_count=1),
            "mean_latency_seconds": pd.to_numeric(group.get("latency_seconds"), errors="coerce").mean(),
        })
        rows.append(row)
    return pd.DataFrame(rows)


def pareto_front(
    ranking: pd.DataFrame,
    maximize: Sequence[str],
    minimize: Sequence[str],
) -> pd.Series:
    if ranking.empty:
        return pd.Series(dtype=bool)
    values = ranking.copy()
    mask = []
    for index, row in values.iterrows():
        dominated = False
        for other_index, other in values.iterrows():
            if index == other_index:
                continue
            weakly_better = all(
                (pd.isna(row[column]) and pd.isna(other[column]))
                or (not pd.isna(other[column]) and (pd.isna(row[column]) or other[column] >= row[column]))
                for column in maximize
            ) and all(
                (pd.isna(row[column]) and pd.isna(other[column]))
                or (not pd.isna(other[column]) and (pd.isna(row[column]) or other[column] <= row[column]))
                for column in minimize
            )
            strictly_better = any(
                not pd.isna(other[column]) and (pd.isna(row[column]) or other[column] > row[column])
                for column in maximize
            ) or any(
                not pd.isna(other[column]) and (pd.isna(row[column]) or other[column] < row[column])
                for column in minimize
            )
            if weakly_better and strictly_better:
                dominated = True
                break
        mask.append(not dominated)
    return pd.Series(mask, index=ranking.index)

In [21]:
def condition_ranking(scored: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    if scored.empty:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
    group_columns = [
        "condition_id", "provider", "definition_id", "prompt_style", "context_id", "workflow"
    ]
    provider_metrics = summarize_binary(scored, group_columns)
    document_metrics = summarize_binary(
        scored,
        ["condition_id", "provider", "document_id", "document_title"],
    )
    document_positive = document_metrics[document_metrics["human_yes"].gt(0)].copy()
    worst_document = (
        document_positive.groupby("condition_id")["recall_yes"]
        .min()
        .rename("worst_provider_document_recall")
        if not document_positive.empty
        else pd.Series(dtype=float)
    )
    ranking = (
        provider_metrics
        .groupby(
            ["condition_id", "definition_id", "prompt_style", "context_id", "workflow"],
            dropna=False,
        )
        .agg(
            providers=("provider", "nunique"),
            mean_accuracy=("accuracy", "mean"),
            mean_balanced_accuracy=("balanced_accuracy", "mean"),
            mean_precision_yes=("precision_yes", "mean"),
            mean_recall_yes=("recall_yes", "mean"),
            mean_specificity=("specificity", "mean"),
            mean_f1_yes=("f1_yes", "mean"),
            mean_mcc=("mcc", "mean"),
            mean_kappa=("cohen_kappa", "mean"),
            schema_valid_rate=("schema_valid_rate", "mean"),
            evidence_valid_strict_rate=("evidence_valid_strict_rate", "mean"),
            evidence_valid_rate=("evidence_valid_rate", "mean"),
            total_cost_usd=("estimated_cost_usd", "sum"),
            mean_latency_seconds=("mean_latency_seconds", "mean"),
        )
        .reset_index()
        .merge(worst_document, on="condition_id", how="left")
    )
    ranking["eligible"] = (
        ranking["providers"].eq(len(EXPECTED_PROVIDERS))
        & ranking["schema_valid_rate"].ge(MIN_SCHEMA_VALID_RATE)
        & ranking["evidence_valid_rate"].ge(MIN_EVIDENCE_VALID_RATE)
        & ranking["mean_specificity"].ge(SPECIFICITY_FLOOR)
    )
    ranking["constraint_shortfall"] = (
        (len(EXPECTED_PROVIDERS) - ranking["providers"]).clip(lower=0)
        + (MIN_SCHEMA_VALID_RATE - ranking["schema_valid_rate"]).clip(lower=0)
        + (MIN_EVIDENCE_VALID_RATE - ranking["evidence_valid_rate"]).clip(lower=0)
        + (SPECIFICITY_FLOOR - ranking["mean_specificity"]).clip(lower=0)
    )
    ranking["pareto_front"] = pareto_front(
        ranking,
        maximize=[
            "worst_provider_document_recall",
            "mean_f1_yes",
            "mean_mcc",
            "mean_specificity",
            "evidence_valid_rate",
        ],
        minimize=["total_cost_usd"],
    )
    ranking = ranking.sort_values(
        [
            "eligible",
            "constraint_shortfall",
            "worst_provider_document_recall",
            "mean_f1_yes",
            "mean_mcc",
            "mean_balanced_accuracy",
            "mean_specificity",
            "total_cost_usd",
            "condition_id",
        ],
        ascending=[False, True, False, False, False, False, False, True, True],
        na_position="last",
    ).reset_index(drop=True)
    ranking["rank"] = np.arange(1, len(ranking) + 1)
    return ranking, provider_metrics, document_metrics


def selection_status(phase: str, ranking: pd.DataFrame) -> dict[str, Any]:
    eligible = ranking[ranking["eligible"]].copy() if not ranking.empty else pd.DataFrame()
    if ranking.empty:
        return {
            "phase": phase,
            "status": "no_results",
            "selected_condition_id": None,
            "selection_basis": "no ranking available",
        }
    if not FINAL_GOLD_COMPLETE:
        return {
            "phase": phase,
            "status": "provisional_historical_gold",
            "selected_condition_id": ranking.iloc[0]["condition_id"],
            "selection_basis": "provisional minimum-shortfall candidate; not a final winner",
            "eligible_conditions": int(ranking["eligible"].sum()),
        }
    if eligible.empty:
        return {
            "phase": phase,
            "status": "no_condition_met_constraints",
            "selected_condition_id": None,
            "minimum_shortfall_candidate": ranking.iloc[0]["condition_id"],
            "selection_basis": "no final winner; inspect Pareto front and revise protocol or thresholds prospectively",
            "eligible_conditions": 0,
        }
    return {
        "phase": phase,
        "status": "final_eligible_winner",
        "selected_condition_id": eligible.iloc[0]["condition_id"],
        "selection_basis": "best preregistered-eligible condition on complete adjudicated final gold",
        "eligible_conditions": len(eligible),
    }

In [22]:
PHASE1_RANKING, PHASE1_PROVIDER_METRICS, PHASE1_DOCUMENT_METRICS = condition_ranking(SCORED["phase1"])
PHASE2_RANKING, PHASE2_PROVIDER_METRICS, PHASE2_DOCUMENT_METRICS = condition_ranking(SCORED["phase2"])
PHASE3_RANKING, PHASE3_PROVIDER_METRICS, PHASE3_DOCUMENT_METRICS = condition_ranking(SCORED["phase3"])

SELECTION_STATUS = pd.DataFrame([
    selection_status("phase1", PHASE1_RANKING),
    selection_status("phase2", PHASE2_RANKING),
    selection_status("phase3", PHASE3_RANKING),
])

display(SELECTION_STATUS)
print("Phase 1 corrected ranking")
display(PHASE1_RANKING)
print("Phase 2 corrected ranking")
display(PHASE2_RANKING)
print("Phase 3 corrected ranking")
display(PHASE3_RANKING)

,phase,status,selected_condition_id,selection_basis,eligible_conditions
0,phase1,provisional_historical_gold,P1_DIRECT,provisional minimum-shortfall candidate; not a final winner,0
1,phase2,provisional_historical_gold,P1_DIRECT__C1_target,provisional minimum-shortfall candidate; not a final winner,0
2,phase3,provisional_historical_gold,W1_ONE_STAGE,provisional minimum-shortfall candidate; not a final winner,0


Phase 1 corrected ranking


,condition_id,definition_id,prompt_style,context_id,workflow,providers,mean_accuracy,mean_balanced_accuracy,mean_precision_yes,mean_recall_yes,mean_specificity,mean_f1_yes,mean_mcc,mean_kappa,schema_valid_rate,evidence_valid_strict_rate,evidence_valid_rate,total_cost_usd,mean_latency_seconds,worst_provider_document_recall,eligible,constraint_shortfall,pareto_front,rank
0,P1_DIRECT,D0_none,P1_direct,C1_target,W1_one_stage,3,0.468254,0.616667,0.933333,0.333333,0.900000,0.485364,0.230461,0.135515,1.0,0.952381,0.976190,0.300032,1.810204,0.166667,False,0.003810,True,1
1,P3_D2_CURRENT,D2_current_strict,P3_evidence_checklist,C1_target,W1_one_stage,3,0.507937,0.665625,0.979167,0.364583,0.966667,0.525794,0.316138,0.197925,1.0,0.896825,0.944444,0.425127,2.307064,0.187500,False,0.035556,True,2
2,P3_D3_ADAPTIVE,D3_provisional_adaptive,P3_evidence_checklist,C1_target,W1_one_stage,3,0.380952,0.582292,0.972222,0.197917,0.966667,0.311146,0.194342,0.091408,1.0,0.904762,0.944444,0.465131,2.398845,0.000000,False,0.035556,False,3
3,P2_D3_ADAPTIVE,D3_provisional_adaptive,P2_simple_definition,C1_target,W1_one_stage,3,0.404762,0.586458,0.952381,0.239583,0.933333,0.364389,0.194773,0.097348,1.0,0.912698,0.944444,0.373258,1.940786,0.000000,False,0.035556,True,4
4,P2_D2_CURRENT,D2_current_strict,P2_simple_definition,C1_target,W1_one_stage,3,0.428571,0.613542,0.969697,0.260417,0.966667,0.408030,0.242339,0.126015,1.0,0.904762,0.920635,0.325021,2.031538,0.062500,False,0.059365,True,5
5,P3_D1_OLD,D1_old_broad,P3_evidence_checklist,C1_target,W1_one_stage,3,0.555556,0.605208,0.847547,0.510417,0.700000,0.635454,0.180975,0.145493,1.0,0.888889,0.944444,0.428266,2.448193,0.400000,False,0.135556,True,6
6,P2_D1_OLD,D1_old_broad,P2_simple_definition,C1_target,W1_one_stage,3,0.571429,0.604167,0.842029,0.541667,0.666667,0.652795,0.180926,0.151540,1.0,0.912698,0.936508,0.343165,1.941406,0.333333,False,0.176825,True,7


Phase 2 corrected ranking


,condition_id,definition_id,prompt_style,context_id,workflow,providers,mean_accuracy,mean_balanced_accuracy,mean_precision_yes,mean_recall_yes,mean_specificity,mean_f1_yes,mean_mcc,mean_kappa,schema_valid_rate,evidence_valid_strict_rate,evidence_valid_rate,total_cost_usd,mean_latency_seconds,worst_provider_document_recall,eligible,constraint_shortfall,pareto_front,rank
0,P1_DIRECT__C1_target,D0_none,P1_direct,C1_target,W1_one_stage,3,0.468254,0.616667,0.933333,0.333333,0.900000,0.485364,0.230461,0.135515,1.0,0.952381,0.976190,0.300032,1.810204,0.166667,False,0.003810,True,1
1,P1_DIRECT__C2_metadata,D0_none,P1_direct,C2_metadata,W1_one_stage,3,0.492063,0.609375,0.903704,0.385417,0.833333,0.531590,0.208071,0.131801,1.0,0.936508,0.952381,0.323340,1.888263,0.166667,False,0.027619,True,2
2,P3_D2_CURRENT__C3_plusminus1,D2_current_strict,P3_evidence_checklist,C3_plusminus1,W1_one_stage,3,0.555556,0.696875,0.984127,0.427083,0.966667,0.580845,0.364237,0.252842,1.0,0.880952,0.952381,0.496904,2.625335,0.125000,False,0.027619,True,3
3,P3_D2_CURRENT__C2_metadata,D2_current_strict,P3_evidence_checklist,C2_metadata,W1_one_stage,3,0.500000,0.660417,0.980392,0.354167,0.966667,0.507888,0.310029,0.193734,1.0,0.904762,0.952381,0.434097,2.571467,0.062500,False,0.027619,True,4
4,P3_D2_CURRENT__C1_target,D2_current_strict,P3_evidence_checklist,C1_target,W1_one_stage,3,0.507937,0.665625,0.979167,0.364583,0.966667,0.525794,0.316138,0.197925,1.0,0.896825,0.944444,0.425127,2.307064,0.187500,False,0.035556,True,5
5,P1_DIRECT__C3_plusminus1,D0_none,P1_direct,C3_plusminus1,W1_one_stage,3,0.571429,0.661458,0.935897,0.489583,0.833333,0.626096,0.295293,0.207342,1.0,0.896825,0.928571,0.394790,2.113408,0.333333,False,0.051429,True,6
6,P3_D2_CURRENT__C4_plusminus2,D2_current_strict,P3_evidence_checklist,C4_plusminus2,W1_one_stage,3,0.571429,0.707292,0.985507,0.447917,0.966667,0.594826,0.382743,0.275993,1.0,0.841270,0.928571,0.539600,2.991171,0.187500,False,0.051429,True,7
7,P1_DIRECT__C4_plusminus2,D0_none,P1_direct,C4_plusminus2,W1_one_stage,3,0.563492,0.621875,0.896296,0.510417,0.733333,0.617739,0.230519,0.166650,1.0,0.873016,0.936508,0.452722,2.289421,0.250000,False,0.110159,True,8


Phase 3 corrected ranking


,condition_id,definition_id,prompt_style,context_id,workflow,providers,mean_accuracy,mean_balanced_accuracy,mean_precision_yes,mean_recall_yes,mean_specificity,mean_f1_yes,mean_mcc,mean_kappa,schema_valid_rate,evidence_valid_strict_rate,evidence_valid_rate,total_cost_usd,mean_latency_seconds,worst_provider_document_recall,eligible,constraint_shortfall,pareto_front,rank
0,W1_ONE_STAGE,D0_none,P1_direct,C1_target,W1_one_stage,3,0.47619,0.621875,0.933333,0.343750,0.900000,0.498372,0.237386,0.142209,1.0,0.936508,0.960317,0.301246,1.813200,0.166667,False,0.019683,True,1
1,W2_BINARY_FIRST,D0_none,P1_direct,C1_target,W2_binary_first,3,0.47619,0.598958,0.895833,0.364583,0.833333,0.516414,0.190022,0.114546,1.0,0.896825,0.936508,0.353014,2.111484,0.166667,False,0.043492,True,2


## 11. Correct completeness reconstruction independent of active phases

In [23]:
PHASE1_REGISTERED_CONDITIONS = [
    "P1_DIRECT",
    "P2_D1_OLD",
    "P2_D2_CURRENT",
    "P2_D3_ADAPTIVE",
    "P3_D1_OLD",
    "P3_D2_CURRENT",
    "P3_D3_ADAPTIVE",
]


def observed_unique_rows(frame: pd.DataFrame, provider: str, condition_id: str, stage: str, seed: int) -> int:
    subset = frame[
        frame.get("provider", pd.Series(dtype=str)).eq(provider)
        & frame.get("condition_id", pd.Series(dtype=str)).eq(condition_id)
        & frame.get("stage", pd.Series(dtype=str)).eq(stage)
        & pd.to_numeric(frame.get("seed"), errors="coerce").eq(seed)
    ]
    return subset.get("row_id", pd.Series(dtype=str)).astype(str).nunique()


def corrected_completeness_table() -> pd.DataFrame:
    rows = []
    n_rows = BENCHMARK["row_id"].nunique()

    phase1_seeds = sorted(pd.to_numeric(PHASE1.get("seed"), errors="coerce").dropna().astype(int).unique())
    for provider, condition_id, seed in itertools.product(
        EXPECTED_PROVIDERS, PHASE1_REGISTERED_CONDITIONS, phase1_seeds
    ):
        observed = observed_unique_rows(PHASE1, provider, condition_id, "joint", seed)
        rows.append({
            "phase": "phase1",
            "provider": provider,
            "condition_id": condition_id,
            "stage": "joint",
            "seed": seed,
            "expected_rows": n_rows,
            "observed_rows": observed,
            "complete": observed == n_rows,
            "source": "raw paid calls",
        })

    phase2_expected_new = [
        f"{base_id}__{context_id}"
        for base_id in phase1_selected_ids
        for context_id in ["C2_metadata", "C3_plusminus1", "C4_plusminus2"]
    ]
    phase2_seeds = sorted(pd.to_numeric(PHASE2_NEW.get("seed"), errors="coerce").dropna().astype(int).unique())
    for provider, condition_id, seed in itertools.product(
        EXPECTED_PROVIDERS, phase2_expected_new, phase2_seeds
    ):
        observed = observed_unique_rows(PHASE2_NEW, provider, condition_id, "joint", seed)
        rows.append({
            "phase": "phase2",
            "provider": provider,
            "condition_id": condition_id,
            "stage": "joint",
            "seed": seed,
            "expected_rows": n_rows,
            "observed_rows": observed,
            "complete": observed == n_rows,
            "source": "raw paid calls",
        })
    for provider, base_id, seed in itertools.product(
        EXPECTED_PROVIDERS, phase1_selected_ids, phase1_seeds
    ):
        condition_id = f"{base_id}__C1_target"
        observed = observed_unique_rows(PHASE2, provider, condition_id, "joint", seed)
        rows.append({
            "phase": "phase2_reused_c1",
            "provider": provider,
            "condition_id": condition_id,
            "stage": "joint",
            "seed": seed,
            "expected_rows": n_rows,
            "observed_rows": observed,
            "complete": observed == n_rows,
            "source": "reused Phase 1 calls",
        })

    raw_phase3 = PHASE_SUCCESS["phase3"]
    phase3_seeds = sorted(pd.to_numeric(raw_phase3.get("seed"), errors="coerce").dropna().astype(int).unique())
    for provider, seed in itertools.product(EXPECTED_PROVIDERS, phase3_seeds):
        for condition_id, stage in [("W1_ONE_STAGE", "joint"), ("W2_BINARY_FIRST", "binary")]:
            observed = observed_unique_rows(raw_phase3, provider, condition_id, stage, seed)
            rows.append({
                "phase": "phase3",
                "provider": provider,
                "condition_id": condition_id,
                "stage": stage,
                "seed": seed,
                "expected_rows": n_rows,
                "observed_rows": observed,
                "complete": observed == n_rows,
                "source": "raw paid calls",
            })
        binary = raw_phase3[
            raw_phase3["provider"].eq(provider)
            & raw_phase3["condition_id"].eq("W2_BINARY_FIRST")
            & raw_phase3["stage"].eq("binary")
            & pd.to_numeric(raw_phase3["seed"], errors="coerce").eq(seed)
        ]
        expected_positive = int(binary.get("out__unlearning_present", pd.Series(dtype=bool)).eq(True).sum())
        target = raw_phase3[
            raw_phase3["provider"].eq(provider)
            & raw_phase3["condition_id"].eq("W2_BINARY_FIRST")
            & raw_phase3["stage"].eq("target")
            & pd.to_numeric(raw_phase3["seed"], errors="coerce").eq(seed)
        ]
        observed_target = target["row_id"].astype(str).nunique()
        rows.append({
            "phase": "phase3",
            "provider": provider,
            "condition_id": "W2_BINARY_FIRST",
            "stage": "target",
            "seed": seed,
            "expected_rows": expected_positive,
            "observed_rows": observed_target,
            "complete": observed_target == expected_positive,
            "source": "conditional raw paid calls",
        })

    raw_stability = PHASE_SUCCESS["stability"]
    stability_seeds = sorted(pd.to_numeric(raw_stability.get("seed"), errors="coerce").dropna().astype(int).unique())
    for provider, seed in itertools.product(EXPECTED_PROVIDERS, stability_seeds):
        subset = raw_stability[
            raw_stability["provider"].eq(provider)
            & pd.to_numeric(raw_stability["seed"], errors="coerce").eq(seed)
        ]
        if subset["stage"].eq("joint").any():
            observed = observed_unique_rows(raw_stability, provider, "FINALIST_STABILITY", "joint", seed)
            rows.append({
                "phase": "stability",
                "provider": provider,
                "condition_id": "FINALIST_STABILITY",
                "stage": "joint",
                "seed": seed,
                "expected_rows": n_rows,
                "observed_rows": observed,
                "complete": observed == n_rows,
                "source": "raw paid calls",
            })
        if subset["stage"].eq("binary").any():
            observed_binary = observed_unique_rows(raw_stability, provider, "FINALIST_STABILITY", "binary", seed)
            rows.append({
                "phase": "stability",
                "provider": provider,
                "condition_id": "FINALIST_STABILITY",
                "stage": "binary",
                "seed": seed,
                "expected_rows": n_rows,
                "observed_rows": observed_binary,
                "complete": observed_binary == n_rows,
                "source": "raw paid calls",
            })
            binary = subset[subset["stage"].eq("binary")]
            expected_positive = int(binary.get("out__unlearning_present", pd.Series(dtype=bool)).eq(True).sum())
            observed_target = subset[subset["stage"].eq("target")]["row_id"].astype(str).nunique()
            rows.append({
                "phase": "stability",
                "provider": provider,
                "condition_id": "FINALIST_STABILITY",
                "stage": "target",
                "seed": seed,
                "expected_rows": expected_positive,
                "observed_rows": observed_target,
                "complete": observed_target == expected_positive,
                "source": "conditional raw paid calls",
            })
    return pd.DataFrame(rows)


CORRECTED_COMPLETENESS = corrected_completeness_table()
COMPLETENESS_SUMMARY = (
    CORRECTED_COMPLETENESS
    .groupby(["phase", "source"], dropna=False)
    .agg(
        groups=("complete", "size"),
        complete_groups=("complete", "sum"),
        expected_rows=("expected_rows", "sum"),
        observed_rows=("observed_rows", "sum"),
        all_complete=("complete", "all"),
    )
    .reset_index()
)

display(COMPLETENESS_SUMMARY)
if STRICT_SOURCE_COMPLETENESS and not CORRECTED_COMPLETENESS["complete"].all():
    display(CORRECTED_COMPLETENESS[~CORRECTED_COMPLETENESS["complete"]])
    raise RuntimeError("Frozen source run is incomplete; corrected final analysis is blocked.")

,phase,source,groups,complete_groups,expected_rows,observed_rows,all_complete
0,phase1,raw paid calls,21,21,882,882,True
1,phase2,raw paid calls,18,18,756,756,True
2,phase2_reused_c1,reused Phase 1 calls,6,6,252,252,True
3,phase3,conditional raw paid calls,3,3,40,40,True
4,phase3,raw paid calls,6,6,252,252,True
5,stability,raw paid calls,15,15,630,630,True


## 12. Definition impact and corrected EPA diagnostics

In [24]:
PHASE1_ALIGNED_METRICS = summarize_binary(
    SCORED["phase1"],
    ["condition_id", "provider", "definition_id", "prompt_style"],
    gold_column="gold_aligned_int",
)
PHASE1_COMMON_METRICS = summarize_binary(
    SCORED["phase1"],
    ["condition_id", "provider", "definition_id", "prompt_style"],
    gold_column="gold_common_int",
)

metric_columns = [
    "condition_id", "provider", "definition_id", "prompt_style",
    "accuracy", "recall_yes", "specificity", "f1_yes", "mcc",
]
PHASE1_DEFINITION_TRADEOFF = PHASE1_ALIGNED_METRICS[metric_columns].merge(
    PHASE1_COMMON_METRICS[metric_columns],
    on=["condition_id", "provider", "definition_id", "prompt_style"],
    suffixes=("_aligned", "_common_final"),
)

PHASE1_EPA_DIAGNOSTICS = summarize_binary(
    SCORED["phase1"][SCORED["phase1"]["is_epa"]],
    ["condition_id", "provider", "definition_id", "prompt_style"],
    gold_column="gold_common_int",
)

DEFINITION_IMPACT = summarize_binary(
    SCORED["phase1"][SCORED["phase1"]["definition_id"].ne("D0_none")],
    ["definition_id", "prompt_style"],
    gold_column="gold_common_int",
)

D2 = SCORED["phase1"][
    SCORED["phase1"]["definition_id"].eq("D2_current_strict")
][["provider", "prompt_style", "row_id", "seed", "pred_int"]].rename(
    columns={"pred_int": "d2_pred"}
)
D3 = SCORED["phase1"][
    SCORED["phase1"]["definition_id"].eq("D3_provisional_adaptive")
][
    ["provider", "prompt_style", "row_id", "seed", "pred_int", "out__change_type", "out__unlearning_mode"]
].rename(columns={
    "pred_int": "d3_pred",
    "out__change_type": "d3_change_type",
    "out__unlearning_mode": "d3_mode",
})

D3_BOUNDARY_CASES = (
    D2.merge(D3, on=["provider", "prompt_style", "row_id", "seed"], how="inner")
    .loc[lambda df: df["d2_pred"].ne(df["d3_pred"])]
    .merge(
        BENCHMARK[[
            "row_id", "document_id", "document_title", "is_epa",
            "target_text", "gold_historical", "gold_final_definition",
        ]],
        on="row_id",
        how="left",
        validate="many_to_one",
    )
)

print("Definition tradeoff")
display(PHASE1_DEFINITION_TRADEOFF)
print("Exact EPA diagnostics")
display(PHASE1_EPA_DIAGNOSTICS)
print("D2/D3 boundary cases with exact EPA identity")
display(D3_BOUNDARY_CASES.sort_values(["is_epa", "row_id"], ascending=[False, True]))

Definition tradeoff


,condition_id,provider,definition_id,prompt_style,accuracy_aligned,recall_yes_aligned,specificity_aligned,f1_yes_aligned,mcc_aligned,accuracy_common_final,recall_yes_common_final,specificity_common_final,f1_yes_common_final,mcc_common_final
0,P1_DIRECT,anthropic,D0_none,P1_direct,0.452381,0.37500,0.7,0.510638,0.066667,0.452381,0.37500,0.7,0.510638,0.066667
1,P1_DIRECT,gemini,D0_none,P1_direct,0.523810,0.37500,1.0,0.545455,0.353553,0.523810,0.37500,1.0,0.545455,0.353553
2,P1_DIRECT,openai,D0_none,P1_direct,0.428571,0.25000,1.0,0.400000,0.271163,0.428571,0.25000,1.0,0.400000,0.271163
3,P2_D1_OLD,anthropic,D1_old_broad,P2_simple_definition,0.595238,0.59375,0.6,0.690909,0.165797,0.595238,0.59375,0.6,0.690909,0.165797
4,P2_D1_OLD,gemini,D1_old_broad,P2_simple_definition,0.500000,0.40625,0.8,0.553191,0.183333,0.500000,0.40625,0.8,0.553191,0.183333
5,P2_D1_OLD,openai,D1_old_broad,P2_simple_definition,0.619048,0.62500,0.6,0.714286,0.193649,0.619048,0.62500,0.6,0.714286,0.193649
6,P2_D2_CURRENT,anthropic,D2_current_strict,P2_simple_definition,0.452381,0.31250,0.9,0.465116,0.205853,0.452381,0.31250,0.9,0.465116,0.205853
7,P2_D2_CURRENT,gemini,D2_current_strict,P2_simple_definition,0.428571,0.25000,1.0,0.400000,0.271163,0.428571,0.25000,1.0,0.400000,0.271163
8,P2_D2_CURRENT,openai,D2_current_strict,P2_simple_definition,0.404762,0.21875,1.0,0.358974,0.250000,0.404762,0.21875,1.0,0.358974,0.250000
9,P2_D3_ADAPTIVE,anthropic,D3_provisional_adaptive,P2_simple_definition,0.476190,0.37500,0.8,0.521739,0.158114,0.476190,0.37500,0.8,0.521739,0.158114


Exact EPA diagnostics


,condition_id,provider,definition_id,prompt_style,n_scored,human_yes,human_no,pred_yes,pred_no,tp,tn,fp,fn,accuracy,balanced_accuracy,precision_yes,recall_yes,specificity,f1_yes,mcc,cohen_kappa,npv,brier_score,schema_valid_rate,evidence_valid_strict_rate,evidence_valid_rate,mean_confidence,input_tokens,output_tokens,estimated_cost_usd,mean_latency_seconds
0,P1_DIRECT,anthropic,D0_none,P1_direct,21,16,5,6,15,5,4,1,11,0.428571,0.55625,0.833333,0.3125,0.8,0.454545,0.106066,0.066667,0.266667,0.510190,1.0,0.857143,0.904762,0.910476,33141,2647,0.046376,1.566804
1,P1_DIRECT,gemini,D0_none,P1_direct,21,16,5,5,16,5,5,0,11,0.476190,0.65625,1.000000,0.3125,1.0,0.476190,0.312500,0.177936,0.312500,0.509167,1.0,0.952381,0.952381,0.954762,11148,3349,0.007810,1.636036
2,P1_DIRECT,openai,D0_none,P1_direct,21,16,5,3,18,3,5,0,13,0.380952,0.59375,1.000000,0.1875,1.0,0.315789,0.228218,0.099010,0.277778,0.547133,1.0,1.000000,1.000000,0.926667,18693,2884,0.090659,2.087288
3,P2_D1_OLD,anthropic,D1_old_broad,P2_simple_definition,21,16,5,11,10,9,3,2,7,0.571429,0.58125,0.818182,0.5625,0.6,0.666667,0.138580,0.120930,0.300000,0.372729,1.0,0.857143,0.904762,0.870952,36165,3130,0.051815,1.770527
4,P2_D1_OLD,gemini,D1_old_broad,P2_simple_definition,21,16,5,7,14,7,5,0,9,0.571429,0.71875,1.000000,0.4375,1.0,0.608696,0.395285,0.270270,0.357143,0.415833,1.0,0.904762,0.904762,0.940476,13920,3522,0.008763,1.321570
5,P2_D1_OLD,openai,D1_old_broad,P2_simple_definition,21,16,5,11,10,9,3,2,7,0.571429,0.58125,0.818182,0.5625,0.6,0.666667,0.138580,0.120930,0.300000,0.379233,1.0,0.952381,0.952381,0.877619,21570,3106,0.106005,3.241593
6,P2_D2_CURRENT,anthropic,D2_current_strict,P2_simple_definition,21,16,5,4,17,3,4,1,13,0.333333,0.49375,0.750000,0.1875,0.8,0.300000,-0.013558,-0.006849,0.235294,0.588386,1.0,0.809524,0.857143,0.928095,36291,2434,0.048461,2.720704
7,P2_D2_CURRENT,gemini,D2_current_strict,P2_simple_definition,21,16,5,1,20,1,5,0,15,0.285714,0.53125,1.000000,0.0625,1.0,0.117647,0.125000,0.030769,0.250000,0.673810,1.0,0.904762,0.904762,0.971429,14004,3569,0.008854,1.345410
8,P2_D2_CURRENT,openai,D2_current_strict,P2_simple_definition,21,16,5,2,19,2,5,0,14,0.333333,0.56250,1.000000,0.1250,1.0,0.222222,0.181369,0.063694,0.263158,0.609005,1.0,0.904762,0.904762,0.939524,21633,2696,0.100027,2.598803
9,P2_D3_ADAPTIVE,anthropic,D3_provisional_adaptive,P2_simple_definition,21,16,5,10,11,8,3,2,8,0.523810,0.55000,0.800000,0.5000,0.6,0.615385,0.085280,0.070796,0.272727,0.406824,1.0,0.809524,0.904762,0.862381,38622,3107,0.054157,2.346590


D2/D3 boundary cases with exact EPA identity


,provider,prompt_style,row_id,seed,d2_pred,d3_pred,d3_change_type,d3_mode,document_id,document_title,is_epa,target_text,gold_historical,gold_final_definition
25,anthropic,P3_evidence_checklist,P002,17,1,0,diagnosis_only,none,epa_katrina_lessons,Lessons Learned: EPA’s Response to Hurricane Katrina,True,"EPA officials told us that, in some instances, coordination could be improved for future disaster responses. Areas where coordination could be improved occurred with respect to...",No,None
24,anthropic,P2_simple_definition,P003,17,0,1,revise_protocol_or_standard,adaptive_reconfiguration,epa_katrina_lessons,Lessons Learned: EPA’s Response to Hurricane Katrina,True,"From our review of wastewater issues, we found at times that parts of EPA were not communicating. We also found that EPA could coordinate its work better with the States and wi...",Yes,None
35,gemini,P2_simple_definition,P004,17,0,1,revise_protocol_or_standard,adaptive_reconfiguration,epa_katrina_lessons,Lessons Learned: EPA’s Response to Hurricane Katrina,True,"Communication is a key component of emergency response. In two instances brought to our attention, EPA visited two municipal wastewater facilities, asked if their facilities ne...",Yes,None
36,gemini,P3_evidence_checklist,P004,17,0,1,revise_protocol_or_standard,adaptive_reconfiguration,epa_katrina_lessons,Lessons Learned: EPA’s Response to Hurricane Katrina,True,"Communication is a key component of emergency response. In two instances brought to our attention, EPA visited two municipal wastewater facilities, asked if their facilities ne...",Yes,None
9,openai,P3_evidence_checklist,P005,17,1,0,add_capacity_only,none,epa_katrina_lessons,Lessons Learned: EPA’s Response to Hurricane Katrina,True,EPA regional officials concurred that tremendous communication issues existed following Hurricane Katrina. Both Regions 4 and 6 indicated they have taken a number of steps to i...,Yes,None
22,anthropic,P2_simple_definition,P005,17,0,1,revise_protocol_or_standard,adaptive_reconfiguration,epa_katrina_lessons,Lessons Learned: EPA’s Response to Hurricane Katrina,True,EPA regional officials concurred that tremendous communication issues existed following Hurricane Katrina. Both Regions 4 and 6 indicated they have taken a number of steps to i...,Yes,None
29,gemini,P2_simple_definition,P009,17,0,1,revise_protocol_or_standard,adaptive_reconfiguration,epa_katrina_lessons,Lessons Learned: EPA’s Response to Hurricane Katrina,True,"• Initially, there were problems in New Orleans with the transport of drinking water in potentially hazardous tanker trucks. The Louisiana Department of Health and Hospitals, w...",Yes,None
15,anthropic,P3_evidence_checklist,P010,17,1,0,diagnosis_only,none,epa_katrina_lessons,Lessons Learned: EPA’s Response to Hurricane Katrina,True,EPA Region 6 officials said initial attempts to coordinate their drinking waterrelated work with the responsible USACE district were unsuccessful in part due to all of the part...,Yes,None
12,anthropic,P2_simple_definition,P011,17,0,1,revise_protocol_or_standard,adaptive_reconfiguration,epa_katrina_lessons,Lessons Learned: EPA’s Response to Hurricane Katrina,True,"We found no evidence that hurricane response activities were negatively impacted by these coordination issues. For example, EPA Regions 4 and 6 used ESF-10 to obtain the necess...",Yes,None
5,openai,P3_evidence_checklist,P014,17,1,0,no_change,none,epa_katrina_lessons,Lessons Learned: EPA’s Response to Hurricane Katrina,True,"According to Region 6 officials, the NPDES Permits and Water Enforcement Branches were jointly charged with putting together an assessment of the wastewater facilities in Louis...",Yes,None


## 13. Context effects and timing caveat

In [25]:
def timestamp_summary(frame: pd.DataFrame) -> pd.DataFrame:
    if frame.empty:
        return pd.DataFrame()
    data = frame.copy()
    data["completed_timestamp"] = pd.to_datetime(data.get("completed_at_utc"), utc=True, errors="coerce")
    return (
        data.groupby(
            ["condition_id", "provider", "context_id", "source_phase"],
            dropna=False,
        )
        .agg(
            rows=("row_id", "size"),
            first_call_utc=("completed_timestamp", "min"),
            last_call_utc=("completed_timestamp", "max"),
            returned_models=("returned_model", lambda x: " | ".join(sorted(set(x.dropna().astype(str))))),
        )
        .reset_index()
    )


PHASE2_TIMING_AUDIT = timestamp_summary(SCORED["phase2"])
PHASE2_CONTEXT_COMPARISON = PHASE2_RANKING.copy()
PHASE2_CONTEXT_COMPARISON["c1_reused_from_phase1"] = PHASE2_CONTEXT_COMPARISON["context_id"].eq("C1_target")
PHASE2_CONTEXT_COMPARISON["timing_interpretation"] = np.where(
    PHASE2_CONTEXT_COMPARISON["c1_reused_from_phase1"],
    "C1 was reused from Phase 1; confirm top context against a contemporaneous C1 rerun before final lock-in.",
    "New Phase 2 context call.",
)

display(PHASE2_CONTEXT_COMPARISON)
display(PHASE2_TIMING_AUDIT)

,condition_id,definition_id,prompt_style,context_id,workflow,providers,mean_accuracy,mean_balanced_accuracy,mean_precision_yes,mean_recall_yes,mean_specificity,mean_f1_yes,mean_mcc,mean_kappa,schema_valid_rate,evidence_valid_strict_rate,evidence_valid_rate,total_cost_usd,mean_latency_seconds,worst_provider_document_recall,eligible,constraint_shortfall,pareto_front,rank,c1_reused_from_phase1,timing_interpretation
0,P1_DIRECT__C1_target,D0_none,P1_direct,C1_target,W1_one_stage,3,0.468254,0.616667,0.933333,0.333333,0.900000,0.485364,0.230461,0.135515,1.0,0.952381,0.976190,0.300032,1.810204,0.166667,False,0.003810,True,1,True,C1 was reused from Phase 1; confirm top context against a contemporaneous C1 rerun before final lock-in.
1,P1_DIRECT__C2_metadata,D0_none,P1_direct,C2_metadata,W1_one_stage,3,0.492063,0.609375,0.903704,0.385417,0.833333,0.531590,0.208071,0.131801,1.0,0.936508,0.952381,0.323340,1.888263,0.166667,False,0.027619,True,2,False,New Phase 2 context call.
2,P3_D2_CURRENT__C3_plusminus1,D2_current_strict,P3_evidence_checklist,C3_plusminus1,W1_one_stage,3,0.555556,0.696875,0.984127,0.427083,0.966667,0.580845,0.364237,0.252842,1.0,0.880952,0.952381,0.496904,2.625335,0.125000,False,0.027619,True,3,False,New Phase 2 context call.
3,P3_D2_CURRENT__C2_metadata,D2_current_strict,P3_evidence_checklist,C2_metadata,W1_one_stage,3,0.500000,0.660417,0.980392,0.354167,0.966667,0.507888,0.310029,0.193734,1.0,0.904762,0.952381,0.434097,2.571467,0.062500,False,0.027619,True,4,False,New Phase 2 context call.
4,P3_D2_CURRENT__C1_target,D2_current_strict,P3_evidence_checklist,C1_target,W1_one_stage,3,0.507937,0.665625,0.979167,0.364583,0.966667,0.525794,0.316138,0.197925,1.0,0.896825,0.944444,0.425127,2.307064,0.187500,False,0.035556,True,5,True,C1 was reused from Phase 1; confirm top context against a contemporaneous C1 rerun before final lock-in.
5,P1_DIRECT__C3_plusminus1,D0_none,P1_direct,C3_plusminus1,W1_one_stage,3,0.571429,0.661458,0.935897,0.489583,0.833333,0.626096,0.295293,0.207342,1.0,0.896825,0.928571,0.394790,2.113408,0.333333,False,0.051429,True,6,False,New Phase 2 context call.
6,P3_D2_CURRENT__C4_plusminus2,D2_current_strict,P3_evidence_checklist,C4_plusminus2,W1_one_stage,3,0.571429,0.707292,0.985507,0.447917,0.966667,0.594826,0.382743,0.275993,1.0,0.841270,0.928571,0.539600,2.991171,0.187500,False,0.051429,True,7,False,New Phase 2 context call.
7,P1_DIRECT__C4_plusminus2,D0_none,P1_direct,C4_plusminus2,W1_one_stage,3,0.563492,0.621875,0.896296,0.510417,0.733333,0.617739,0.230519,0.166650,1.0,0.873016,0.936508,0.452722,2.289421,0.250000,False,0.110159,True,8,False,New Phase 2 context call.


,condition_id,provider,context_id,source_phase,rows,first_call_utc,last_call_utc,returned_models
0,P1_DIRECT__C1_target,anthropic,C1_target,phase1_reused,42,2026-07-22 20:58:12.488870+00:00,2026-07-22 21:09:08.574487+00:00,claude-haiku-4-5-20251001
1,P1_DIRECT__C1_target,gemini,C1_target,phase1_reused,42,2026-07-22 21:09:30.457494+00:00,2026-07-22 21:39:36.048229+00:00,gemini-3.1-flash-lite
2,P1_DIRECT__C1_target,openai,C1_target,phase1_reused,42,2026-07-22 20:44:42.592711+00:00,2026-07-22 20:57:38.058238+00:00,gpt-5.6-terra
3,P1_DIRECT__C2_metadata,anthropic,C2_metadata,phase2_new_call,42,2026-07-22 22:32:00.966428+00:00,2026-07-22 22:59:57.676041+00:00,claude-haiku-4-5-20251001
4,P1_DIRECT__C2_metadata,gemini,C2_metadata,phase2_new_call,42,2026-07-22 23:01:49.674153+00:00,2026-07-23 22:40:03.868279+00:00,gemini-3.1-flash-lite
5,P1_DIRECT__C2_metadata,openai,C2_metadata,phase2_new_call,42,2026-07-22 22:01:09.305911+00:00,2026-07-22 22:30:01.873296+00:00,gpt-5.6-terra
6,P1_DIRECT__C3_plusminus1,anthropic,C3_plusminus1,phase2_new_call,42,2026-07-22 22:31:32.361233+00:00,2026-07-22 23:00:32.604781+00:00,claude-haiku-4-5-20251001
7,P1_DIRECT__C3_plusminus1,gemini,C3_plusminus1,phase2_new_call,42,2026-07-22 23:00:52.268931+00:00,2026-07-23 22:40:29.606346+00:00,gemini-3.1-flash-lite
8,P1_DIRECT__C3_plusminus1,openai,C3_plusminus1,phase2_new_call,42,2026-07-22 22:00:40.388154+00:00,2026-07-22 22:30:30.226870+00:00,gpt-5.6-terra
9,P1_DIRECT__C4_plusminus2,anthropic,C4_plusminus2,phase2_new_call,42,2026-07-22 22:31:40.260900+00:00,2026-07-22 23:00:04.483556+00:00,claude-haiku-4-5-20251001


## 14. Target-taxonomy validity audit

Historical target cells may contain more than one human code (for example,
`Laws, plans and policies; Leadership`). This notebook parses those cells into
sets and reports **membership accuracy** rather than falsely treating the whole
string as one category. A final single-label target result remains provisional
until adjudication assigns exactly one canonical target to each final-positive row.

In [26]:
TARGET_ALLOWED = {
    "leadership",
    "laws_plans_policies",
    "capabilities",
    "funds_resources",
    "misc_organizational",
    "none",
}
TARGET_POSITIVE = TARGET_ALLOWED - {"none"}


def parse_target_set(value: Any) -> frozenset[str]:
    """Parse historical single- or multi-label target cells into canonical codes.

    Commas inside the official label ``Laws, plans and policies`` are not treated
    as label separators. Semicolon-separated human labels are retained as sets.
    """
    raw = normalize_for_match(value)
    if not raw or raw in {"na", "n/a", "none", "not applicable", "nan"}:
        return frozenset()

    # Normalize punctuation but preserve all words so that multiple categories
    # can be detected in a single historical cell.
    text = raw.replace("_", " ").replace("/", " ")
    text = re.sub(r"[^a-z0-9]+", " ", text)
    text = normalize_space(text)
    matches: set[str] = set()

    if re.search(r"\bleadership\b", text):
        matches.add("leadership")
    if (
        re.search(r"\blaws?\b", text)
        and re.search(r"\bplans?\b", text)
        and re.search(r"\bpolic(?:y|ies)\b", text)
    ):
        matches.add("laws_plans_policies")
    if re.search(r"\bcapabilit(?:y|ies)\b", text):
        matches.add("capabilities")
    if (
        re.search(r"\bfunds?\b", text)
        or re.search(r"\bresource(?:s| allocation)?\b", text)
    ) and (
        "allocation" in text
        or "fund" in text
        or "resource" in text
    ):
        matches.add("funds_resources")
    if re.search(r"\bmisc(?:ellaneous)?\b", text) or "misc organizational" in text:
        matches.add("misc_organizational")

    # Model outputs use an explicit none category. Historical blank/NA cells are
    # represented by an empty set, which is equivalent for negative rows.
    if not matches and text == "none":
        matches.add("none")
    return frozenset(matches)


def normalize_target(value: Any) -> Optional[str]:
    parsed = parse_target_set(value)
    return next(iter(parsed)) if len(parsed) == 1 else None


def serialized_target_set(value: Any) -> str:
    return " | ".join(sorted(parse_target_set(value)))


TARGET_TAXONOMY_AUDIT = BENCHMARK[[
    "row_id", "document_id", "document_title", "gold_historical",
    "gold_final_definition", "gold_target", "target_text",
]].copy()
TARGET_TAXONOMY_AUDIT["gold_for_target_audit"] = (
    TARGET_TAXONOMY_AUDIT[COMMON_FINAL_COLUMN].map(normalize_label)
)
TARGET_TAXONOMY_AUDIT["gold_target_set"] = (
    TARGET_TAXONOMY_AUDIT["gold_target"].map(serialized_target_set)
)
TARGET_TAXONOMY_AUDIT["gold_target_count"] = (
    TARGET_TAXONOMY_AUDIT["gold_target"].map(lambda value: len(parse_target_set(value)))
)
TARGET_TAXONOMY_AUDIT["potential_multilabel"] = (
    TARGET_TAXONOMY_AUDIT["gold_target_count"].gt(1)
)
TARGET_TAXONOMY_AUDIT["unmapped_nonempty_target"] = (
    TARGET_TAXONOMY_AUDIT["gold_target"].map(normalize_space).ne("")
    & TARGET_TAXONOMY_AUDIT["gold_target_count"].eq(0)
    & ~TARGET_TAXONOMY_AUDIT["gold_target"].map(normalize_for_match).isin(
        {"na", "n/a", "none", "not applicable", "nan"}
    )
)
TARGET_TAXONOMY_AUDIT["positive_missing_or_unmapped_target"] = (
    TARGET_TAXONOMY_AUDIT["gold_for_target_audit"].eq("Yes")
    & ~TARGET_TAXONOMY_AUDIT["gold_target"].map(
        lambda value: bool(parse_target_set(value) & TARGET_POSITIVE)
    )
)
TARGET_TAXONOMY_AUDIT["negative_has_positive_target"] = (
    TARGET_TAXONOMY_AUDIT["gold_for_target_audit"].eq("No")
    & TARGET_TAXONOMY_AUDIT["gold_target"].map(
        lambda value: bool(parse_target_set(value) & TARGET_POSITIVE)
    )
)

# Multi-label historical targets are now valid for membership scoring. They are
# still placed in the adjudication queue because the final framework requires one
# canonical target per positive paragraph.
TARGET_EVALUATION_READY = not (
    TARGET_TAXONOMY_AUDIT["positive_missing_or_unmapped_target"].any()
    or TARGET_TAXONOMY_AUDIT["negative_has_positive_target"].any()
    or TARGET_TAXONOMY_AUDIT["unmapped_nonempty_target"].any()
)
TARGET_FINAL_SINGLE_LABEL_READY = bool(
    TARGET_EVALUATION_READY
    and not TARGET_TAXONOMY_AUDIT.loc[
        TARGET_TAXONOMY_AUDIT["gold_for_target_audit"].eq("Yes"),
        "potential_multilabel",
    ].any()
)

TARGET_TAXONOMY_SUMMARY = pd.DataFrame([{
    "target_membership_evaluation_ready": TARGET_EVALUATION_READY,
    "final_single_label_target_ready": TARGET_FINAL_SINGLE_LABEL_READY,
    "potential_multilabel_rows": int(TARGET_TAXONOMY_AUDIT["potential_multilabel"].sum()),
    "positive_missing_or_unmapped": int(TARGET_TAXONOMY_AUDIT["positive_missing_or_unmapped_target"].sum()),
    "negative_with_positive_target": int(TARGET_TAXONOMY_AUDIT["negative_has_positive_target"].sum()),
    "unmapped_nonempty_target": int(TARGET_TAXONOMY_AUDIT["unmapped_nonempty_target"].sum()),
    "unique_raw_gold_targets": int(BENCHMARK["gold_target"].nunique(dropna=True)),
    "interpretation": (
        "Historical multi-label targets are scored by set membership; final exact "
        "target inference remains blocked until adjudication assigns one target."
    ),
}])

display(TARGET_TAXONOMY_SUMMARY)
display(TARGET_TAXONOMY_AUDIT[
    TARGET_TAXONOMY_AUDIT[[
        "potential_multilabel",
        "unmapped_nonempty_target",
        "positive_missing_or_unmapped_target",
        "negative_has_positive_target",
    ]].any(axis=1)
])

,target_membership_evaluation_ready,final_single_label_target_ready,potential_multilabel_rows,positive_missing_or_unmapped,negative_with_positive_target,unmapped_nonempty_target,unique_raw_gold_targets,interpretation
0,False,False,1,26,0,0,4,Historical multi-label targets are scored by set membership; final exact target inference remains blocked until adjudication assigns one target.


,row_id,document_id,document_title,gold_historical,gold_final_definition,gold_target,target_text,gold_for_target_audit,gold_target_set,gold_target_count,potential_multilabel,unmapped_nonempty_target,positive_missing_or_unmapped_target,negative_has_positive_target
2,P003,epa_katrina_lessons,Lessons Learned: EPA’s Response to Hurricane Katrina,Yes,None,None,"From our review of wastewater issues, we found at times that parts of EPA were not communicating. We also found that EPA could coordinate its work better with the States and wi...",Yes,,0,False,False,True,False
3,P004,epa_katrina_lessons,Lessons Learned: EPA’s Response to Hurricane Katrina,Yes,None,None,"Communication is a key component of emergency response. In two instances brought to our attention, EPA visited two municipal wastewater facilities, asked if their facilities ne...",Yes,,0,False,False,True,False
4,P005,epa_katrina_lessons,Lessons Learned: EPA’s Response to Hurricane Katrina,Yes,None,None,EPA regional officials concurred that tremendous communication issues existed following Hurricane Katrina. Both Regions 4 and 6 indicated they have taken a number of steps to i...,Yes,,0,False,False,True,False
5,P006,epa_katrina_lessons,Lessons Learned: EPA’s Response to Hurricane Katrina,Yes,None,None,"This memorandum addresses the fifth and final question of our drinking water evaluation – whether EPA followed its emergency response protocols, including those lessons learned...",Yes,,0,False,False,True,False
7,P008,epa_katrina_lessons,Lessons Learned: EPA’s Response to Hurricane Katrina,Yes,None,None,"• Coordination within EPA, as well as EPA’s coordination with State and local officials and the USACE, could have been better. In some instances, coordination problems resulted...",Yes,,0,False,False,True,False
8,P009,epa_katrina_lessons,Lessons Learned: EPA’s Response to Hurricane Katrina,Yes,None,None,"• Initially, there were problems in New Orleans with the transport of drinking water in potentially hazardous tanker trucks. The Louisiana Department of Health and Hospitals, w...",Yes,,0,False,False,True,False
9,P010,epa_katrina_lessons,Lessons Learned: EPA’s Response to Hurricane Katrina,Yes,None,None,EPA Region 6 officials said initial attempts to coordinate their drinking waterrelated work with the responsible USACE district were unsuccessful in part due to all of the part...,Yes,,0,False,False,True,False
10,P011,epa_katrina_lessons,Lessons Learned: EPA’s Response to Hurricane Katrina,Yes,None,None,"We found no evidence that hurricane response activities were negatively impacted by these coordination issues. For example, EPA Regions 4 and 6 used ESF-10 to obtain the necess...",Yes,,0,False,False,True,False
11,P012,epa_katrina_lessons,Lessons Learned: EPA’s Response to Hurricane Katrina,Yes,None,None,". . . develop integrated operational plans, procedures and capabilities for their support to the base NRP and all ESFs and Support Annexes…. Each primary department or Agency f...",Yes,,0,False,False,True,False
12,P013,epa_katrina_lessons,Lessons Learned: EPA’s Response to Hurricane Katrina,Yes,None,None,"Region 4 officials concurred that there is a need for EPA to play a more direct role in responding with USACE to water issues under ESF-3, similar to what is currently done wit...",Yes,,0,False,False,True,False


In [27]:
def agency_token_set(value: Any) -> set[str]:
    return {
        token for token in re.findall(r"[a-z0-9]+", normalize_for_match(value))
        if token not in {
            "the", "of", "and", "department", "office", "agency", "government"
        }
    }


def target_agency_metrics(data: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    if data.empty or not TARGET_EVALUATION_READY:
        status = pd.DataFrame([{
            "status": "not_valid_for_inference",
            "reason": (
                "One or more positive rows lack a mappable target, a negative row "
                "has a positive target, or a nonempty target value is unmapped."
            ),
        }])
        return status, pd.DataFrame()

    scored = data.copy()
    scored["gold_target_set_obj"] = scored["gold_target"].map(parse_target_set)
    scored["gold_target_count"] = scored["gold_target_set_obj"].map(len)
    scored["pred_target_norm"] = scored.get("out__target_type").map(normalize_target)
    gold_positive = scored["gold_common_int"].eq(1)

    scored["target_membership"] = scored.apply(
        lambda row: (
            np.nan
            if row["gold_common_int"] != 1
            else row.get("pred_target_norm") in row["gold_target_set_obj"]
        ),
        axis=1,
    )
    scored["target_single_label_exact"] = scored.apply(
        lambda row: (
            np.nan
            if row["gold_common_int"] != 1 or row["gold_target_count"] != 1
            else row.get("pred_target_norm") in row["gold_target_set_obj"]
        ),
        axis=1,
    )
    scored["target_jaccard"] = scored.apply(
        lambda row: (
            np.nan
            if row["gold_common_int"] != 1 or not row["gold_target_set_obj"]
            else (
                1.0 / len(row["gold_target_set_obj"])
                if row.get("pred_target_norm") in row["gold_target_set_obj"]
                else 0.0
            )
        ),
        axis=1,
    )
    scored["binary_plus_target_membership"] = scored.apply(
        lambda row: bool(
            row["gold_common_int"] == row["pred_int"]
            and (
                row["gold_common_int"] == 0
                or row.get("pred_target_norm") in row["gold_target_set_obj"]
            )
        ),
        axis=1,
    )
    scored["agency_overlap"] = scored.apply(
        lambda row: (
            np.nan
            if row["gold_common_int"] != 1 or not normalize_space(row.get("gold_agency"))
            else bool(
                agency_token_set(row.get("gold_agency"))
                & agency_token_set(row.get("out__agency"))
            )
        ),
        axis=1,
    )

    summary = (
        scored.groupby(["condition_id", "provider", "workflow"], dropna=False)
        .agg(
            rows=("row_id", "count"),
            gold_positive_rows=("gold_common_int", lambda values: int(pd.Series(values).eq(1).sum())),
            multi_label_gold_rows=("gold_target_count", lambda values: int(pd.Series(values).gt(1).sum())),
            target_membership_accuracy=("target_membership", "mean"),
            target_single_label_accuracy=("target_single_label_exact", "mean"),
            target_mean_jaccard=("target_jaccard", "mean"),
            binary_plus_target_membership_exact=("binary_plus_target_membership", "mean"),
            agency_overlap_accuracy=("agency_overlap", "mean"),
            workflow_complete_rate=("workflow_complete", "mean"),
            stage2_call_rate=("stage2_called", "mean"),
        )
        .reset_index()
    )

    per_category_rows = []
    for _, row in scored[gold_positive].iterrows():
        for category in sorted(row["gold_target_set_obj"] & TARGET_POSITIVE):
            per_category_rows.append({
                "condition_id": row.get("condition_id"),
                "provider": row.get("provider"),
                "workflow": row.get("workflow"),
                "row_id": row.get("row_id"),
                "gold_category": category,
                "detected": row.get("pred_target_norm") == category,
            })
    expanded = pd.DataFrame(per_category_rows)
    per_category = (
        expanded.groupby(
            ["condition_id", "provider", "workflow", "gold_category"],
            dropna=False,
        )
        .agg(gold_instances=("row_id", "size"), recall=("detected", "mean"))
        .reset_index()
        if not expanded.empty
        else pd.DataFrame()
    )
    return summary, per_category


PHASE3_TARGET_METRICS, TARGET_PER_CATEGORY_RECALL = target_agency_metrics(
    SCORED["phase3"]
)
display(PHASE3_TARGET_METRICS)
display(TARGET_PER_CATEGORY_RECALL)

unknown_target_values = sorted(set(
    BENCHMARK.loc[
        BENCHMARK["gold_target"].map(normalize_space).ne("")
        & BENCHMARK["gold_target"].map(lambda value: len(parse_target_set(value)) == 0)
        & ~BENCHMARK["gold_target"].map(normalize_for_match).isin(
            {"na", "n/a", "none", "not applicable", "nan"}
        ),
        "gold_target",
    ].astype(str)
))
TARGET_MAPPING_TEMPLATE = pd.DataFrame({
    "raw_gold_target": unknown_target_values,
    "proposed_canonical_target": "",
    "adjudication_note": "",
})
TARGET_MAPPING_TEMPLATE.to_csv(
    ADJUDICATION_ROOT / "target_taxonomy_mapping_template.csv",
    index=False,
)

,status,reason
0,not_valid_for_inference,"One or more positive rows lack a mappable target, a negative row has a positive target, or a nonempty target value is unmapped."


""


## 15. Stability, tiered review, and reliability

In [28]:
def fleiss_kappa_binary(vote_matrix: np.ndarray) -> float:
    matrix = np.asarray(vote_matrix, dtype=float)
    n_items = matrix.shape[0]
    n_raters = matrix.sum(axis=1)
    if n_items == 0 or np.any(n_raters < 2) or not np.allclose(n_raters, n_raters[0]):
        return np.nan
    n = n_raters[0]
    category_proportions = matrix.sum(axis=0) / (n_items * n)
    expected_agreement = np.sum(category_proportions ** 2)
    item_agreement = (np.sum(matrix ** 2, axis=1) - n) / (n * (n - 1))
    observed_agreement = item_agreement.mean()
    return (
        float((observed_agreement - expected_agreement) / (1 - expected_agreement))
        if expected_agreement < 1
        else np.nan
    )


def krippendorff_alpha_nominal(reliability_data: np.ndarray) -> float:
    """Krippendorff's alpha for nominal values with optional missing entries.

    The implementation follows the coincidence-matrix formulation and is used
    even when the optional third-party package is unavailable.
    """
    matrix = np.asarray(reliability_data, dtype=float)
    categories = sorted({
        int(value)
        for value in matrix.ravel()
        if not np.isnan(value)
    })
    if len(categories) < 2:
        return np.nan
    category_index = {category: index for index, category in enumerate(categories)}
    coincidence = np.zeros((len(categories), len(categories)), dtype=float)

    for unit_values in matrix.T:
        values = [int(value) for value in unit_values if not np.isnan(value)]
        n_unit = len(values)
        if n_unit < 2:
            continue
        counts = Counter(values)
        for left_category, left_count in counts.items():
            left_index = category_index[left_category]
            for right_category, right_count in counts.items():
                right_index = category_index[right_category]
                pair_count = (
                    left_count * (right_count - 1)
                    if left_category == right_category
                    else left_count * right_count
                )
                coincidence[left_index, right_index] += pair_count / (n_unit - 1)

    total_coincidences = coincidence.sum()
    if total_coincidences <= 1:
        return np.nan
    observed_disagreement = (coincidence.sum() - np.trace(coincidence)) / total_coincidences
    marginals = coincidence.sum(axis=1)
    expected_disagreement_numerator = 0.0
    for left_index in range(len(categories)):
        for right_index in range(len(categories)):
            if left_index != right_index:
                expected_disagreement_numerator += (
                    marginals[left_index] * marginals[right_index]
                ) / (total_coincidences - 1)
    expected_disagreement = expected_disagreement_numerator / total_coincidences
    if expected_disagreement <= 0:
        return np.nan
    return float(1 - observed_disagreement / expected_disagreement)


def cross_provider_reliability(data: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for seed, group in data.groupby("seed"):
        pivot = group.pivot_table(
            index="row_id",
            columns="provider",
            values="pred_int",
            aggfunc="first",
        )
        complete = pivot.dropna()
        if complete.empty:
            continue
        counts = np.column_stack([
            (complete == 0).sum(axis=1),
            (complete == 1).sum(axis=1),
        ])
        alpha = krippendorff_alpha_nominal(complete.to_numpy().T)
        package_alpha = np.nan
        if krippendorff_package is not None:
            package_alpha = float(krippendorff_package.alpha(
                reliability_data=complete.to_numpy().T,
                level_of_measurement="nominal",
            ))
            if not np.isclose(alpha, package_alpha, atol=1e-10, equal_nan=True):
                raise AssertionError(
                    "Internal Krippendorff alpha does not match the optional package."
                )
        rows.append({
            "seed": int(seed),
            "rows_complete": len(complete),
            "providers": len(complete.columns),
            "unanimous_agreement": complete.nunique(axis=1).eq(1).mean(),
            "fleiss_kappa": fleiss_kappa_binary(counts),
            "krippendorff_alpha": alpha,
            "krippendorff_package_crosscheck": package_alpha,
        })
    return pd.DataFrame(rows)


def provider_repeat_stability(data: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    rows = []
    pairwise = []
    for (provider, row_id), group in data.groupby(["provider", "row_id"]):
        values = group.sort_values("seed")["pred_int"].dropna().astype(int).tolist()
        counts = Counter(values)
        modal_count = max(counts.values()) if counts else 0
        rows.append({
            "provider": provider,
            "row_id": row_id,
            "n_repeats": len(values),
            "yes_repeats": counts.get(1, 0),
            "no_repeats": counts.get(0, 0),
            "repeat_agreement_share": safe_rate(modal_count, len(values)),
            "changed_across_repeats": len(counts) > 1,
        })
    for provider, provider_data in data.groupby("provider"):
        pivot = provider_data.pivot_table(index="row_id", columns="seed", values="pred_int", aggfunc="first")
        for left_seed, right_seed in itertools.combinations(pivot.columns, 2):
            paired = pivot[[left_seed, right_seed]].dropna()
            if paired.empty:
                continue
            pairwise.append({
                "provider": provider,
                "seed_left": int(left_seed),
                "seed_right": int(right_seed),
                "n_rows": len(paired),
                "percent_agreement": paired[left_seed].eq(paired[right_seed]).mean(),
                "cohen_kappa": (
                    cohen_kappa_score(paired[left_seed], paired[right_seed])
                    if paired[left_seed].nunique() > 1 or paired[right_seed].nunique() > 1
                    else np.nan
                ),
            })
    return pd.DataFrame(rows), pd.DataFrame(pairwise)


STABILITY_RELIABILITY = cross_provider_reliability(SCORED["stability"])
PROVIDER_ROW_STABILITY, PROVIDER_SEED_PAIRWISE = provider_repeat_stability(SCORED["stability"])

display(STABILITY_RELIABILITY)
display(PROVIDER_SEED_PAIRWISE)

,seed,rows_complete,providers,unanimous_agreement,fleiss_kappa,krippendorff_alpha,krippendorff_package_crosscheck
0,17,42,3,0.690476,0.502581,0.506529,NaN
1,43,42,3,0.690476,0.502581,0.506529,NaN
2,101,42,3,0.738095,0.572222,0.575617,NaN
3,211,42,3,0.738095,0.564835,0.568289,NaN
4,307,42,3,0.690476,0.502581,0.506529,NaN


,provider,seed_left,seed_right,n_rows,percent_agreement,cohen_kappa
0,anthropic,17,43,42,1.000000,1.000000
1,anthropic,17,101,42,1.000000,1.000000
2,anthropic,17,211,42,1.000000,1.000000
3,anthropic,17,307,42,1.000000,1.000000
4,anthropic,43,101,42,1.000000,1.000000
5,anthropic,43,211,42,1.000000,1.000000
6,anthropic,43,307,42,1.000000,1.000000
7,anthropic,101,211,42,1.000000,1.000000
8,anthropic,101,307,42,1.000000,1.000000
9,anthropic,211,307,42,1.000000,1.000000


In [29]:
def tier_from_yes_votes(yes_votes: int, n_votes: int) -> str:
    if n_votes != 3:
        return "INCOMPLETE"
    return {3: "Tier 1", 2: "Tier 2", 1: "Tier 3", 0: "Tier 4"}[int(yes_votes)]


def build_tiered_review(data: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    rows = []
    for (row_id, seed), group in data.groupby(["row_id", "seed"]):
        valid = group.dropna(subset=["pred_int"])
        yes_votes = int(valid["pred_int"].sum())
        n_votes = valid["provider"].nunique()
        rows.append({
            "row_id": row_id,
            "seed": int(seed),
            "n_provider_votes": n_votes,
            "yes_votes": yes_votes,
            "no_votes": n_votes - yes_votes,
            "tier": tier_from_yes_votes(yes_votes, n_votes),
            "majority_prediction": "Yes" if yes_votes >= 2 else "No",
            "providers_yes": " | ".join(sorted(valid.loc[valid["pred_int"].eq(1), "provider"].astype(str))),
            "providers_no": " | ".join(sorted(valid.loc[valid["pred_int"].eq(0), "provider"].astype(str))),
            "mean_confidence": valid["confidence"].mean(),
            "any_evidence_invalid": not bool(valid["evidence_valid_normalized"].fillna(False).all()),
        })
    by_seed = pd.DataFrame(rows).merge(
        BENCHMARK[[
            "row_id", "document_id", "document_title", "is_epa",
            "target_text", "gold_historical", "gold_final_definition",
        ]],
        on="row_id",
        how="left",
        validate="many_to_one",
    )
    stability_rows = []
    for row_id, group in by_seed.groupby("row_id"):
        tiers = group.sort_values("seed")["tier"].tolist()
        counts = Counter(tiers)
        stability_rows.append({
            "row_id": row_id,
            "n_repeats": len(tiers),
            "tiers_by_seed": " | ".join(tiers),
            "distinct_tiers": len(counts),
            "modal_tier": counts.most_common(1)[0][0],
            "tier_stable": len(counts) == 1,
        })
    tier_stability = pd.DataFrame(stability_rows).merge(
        BENCHMARK[["row_id", "document_id", "document_title", "is_epa", "target_text"]],
        on="row_id",
        how="left",
        validate="one_to_one",
    )
    return by_seed, tier_stability


TIERED_REVIEW_BY_SEED, TIER_STABILITY = build_tiered_review(SCORED["stability"])
display(TIERED_REVIEW_BY_SEED.head(30))
display(TIER_STABILITY.sort_values(["tier_stable", "row_id"]))

,row_id,seed,n_provider_votes,yes_votes,no_votes,tier,majority_prediction,providers_yes,providers_no,mean_confidence,any_evidence_invalid,document_id,document_title,is_epa,target_text,gold_historical,gold_final_definition
0,P001,17,3,0,3,Tier 4,No,,anthropic | gemini | openai,0.963333,False,epa_katrina_lessons,Lessons Learned: EPA’s Response to Hurricane Katrina,True,"• Coordination within EPA, with State and local officials, and with the U.S. Army Corps of Engineers (USACE) could have been better. In some instances, coordination problems re...",No,None
1,P001,43,3,0,3,Tier 4,No,,anthropic | gemini | openai,0.973333,False,epa_katrina_lessons,Lessons Learned: EPA’s Response to Hurricane Katrina,True,"• Coordination within EPA, with State and local officials, and with the U.S. Army Corps of Engineers (USACE) could have been better. In some instances, coordination problems re...",No,None
2,P001,101,3,0,3,Tier 4,No,,anthropic | gemini | openai,0.976667,False,epa_katrina_lessons,Lessons Learned: EPA’s Response to Hurricane Katrina,True,"• Coordination within EPA, with State and local officials, and with the U.S. Army Corps of Engineers (USACE) could have been better. In some instances, coordination problems re...",No,None
3,P001,211,3,0,3,Tier 4,No,,anthropic | gemini | openai,0.963333,False,epa_katrina_lessons,Lessons Learned: EPA’s Response to Hurricane Katrina,True,"• Coordination within EPA, with State and local officials, and with the U.S. Army Corps of Engineers (USACE) could have been better. In some instances, coordination problems re...",No,None
4,P001,307,3,0,3,Tier 4,No,,anthropic | gemini | openai,0.966667,False,epa_katrina_lessons,Lessons Learned: EPA’s Response to Hurricane Katrina,True,"• Coordination within EPA, with State and local officials, and with the U.S. Army Corps of Engineers (USACE) could have been better. In some instances, coordination problems re...",No,None
5,P002,17,3,1,2,Tier 3,No,anthropic,gemini | openai,0.903333,True,epa_katrina_lessons,Lessons Learned: EPA’s Response to Hurricane Katrina,True,"EPA officials told us that, in some instances, coordination could be improved for future disaster responses. Areas where coordination could be improved occurred with respect to...",No,None
6,P002,43,3,1,2,Tier 3,No,anthropic,gemini | openai,0.903333,True,epa_katrina_lessons,Lessons Learned: EPA’s Response to Hurricane Katrina,True,"EPA officials told us that, in some instances, coordination could be improved for future disaster responses. Areas where coordination could be improved occurred with respect to...",No,None
7,P002,101,3,1,2,Tier 3,No,anthropic,gemini | openai,0.906667,True,epa_katrina_lessons,Lessons Learned: EPA’s Response to Hurricane Katrina,True,"EPA officials told us that, in some instances, coordination could be improved for future disaster responses. Areas where coordination could be improved occurred with respect to...",No,None
8,P002,211,3,1,2,Tier 3,No,anthropic,gemini | openai,0.903333,True,epa_katrina_lessons,Lessons Learned: EPA’s Response to Hurricane Katrina,True,"EPA officials told us that, in some instances, coordination could be improved for future disaster responses. Areas where coordination could be improved occurred with respect to...",No,None
9,P002,307,3,1,2,Tier 3,No,anthropic,gemini | openai,0.903333,True,epa_katrina_lessons,Lessons Learned: EPA’s Response to Hurricane Katrina,True,"EPA officials told us that, in some instances, coordination could be improved for future disaster responses. Areas where coordination could be improved occurred with respect to...",No,None


,row_id,n_repeats,tiers_by_seed,distinct_tiers,modal_tier,tier_stable,document_id,document_title,is_epa,target_text
8,P009,5,Tier 3 | Tier 3 | Tier 2 | Tier 3 | Tier 3,2,Tier 3,False,epa_katrina_lessons,Lessons Learned: EPA’s Response to Hurricane Katrina,True,"• Initially, there were problems in New Orleans with the transport of drinking water in potentially hazardous tanker trucks. The Louisiana Department of Health and Hospitals, w..."
13,P014,5,Tier 3 | Tier 3 | Tier 4 | Tier 4 | Tier 3,2,Tier 3,False,epa_katrina_lessons,Lessons Learned: EPA’s Response to Hurricane Katrina,True,"According to Region 6 officials, the NPDES Permits and Water Enforcement Branches were jointly charged with putting together an assessment of the wastewater facilities in Louis..."
22,P023,5,Tier 3 | Tier 3 | Tier 4 | Tier 4 | Tier 3,2,Tier 3,False,hurricane_katrina_gao_s_preliminary_observations_regarding_preparedness_response,"Hurricane Katrina: GAO’s Preliminary Observations Regarding Preparedness, Response, and Recovery",False,"In addition, we have done a great deal of work on prior disasters. In 1993, we conducted several reviews examining the federal response to Hurricane Andrew. The reviews focused..."
0,P001,5,Tier 4 | Tier 4 | Tier 4 | Tier 4 | Tier 4,1,Tier 4,True,epa_katrina_lessons,Lessons Learned: EPA’s Response to Hurricane Katrina,True,"• Coordination within EPA, with State and local officials, and with the U.S. Army Corps of Engineers (USACE) could have been better. In some instances, coordination problems re..."
1,P002,5,Tier 3 | Tier 3 | Tier 3 | Tier 3 | Tier 3,1,Tier 3,True,epa_katrina_lessons,Lessons Learned: EPA’s Response to Hurricane Katrina,True,"EPA officials told us that, in some instances, coordination could be improved for future disaster responses. Areas where coordination could be improved occurred with respect to..."
2,P003,5,Tier 3 | Tier 3 | Tier 3 | Tier 3 | Tier 3,1,Tier 3,True,epa_katrina_lessons,Lessons Learned: EPA’s Response to Hurricane Katrina,True,"From our review of wastewater issues, we found at times that parts of EPA were not communicating. We also found that EPA could coordinate its work better with the States and wi..."
3,P004,5,Tier 2 | Tier 2 | Tier 2 | Tier 2 | Tier 2,1,Tier 2,True,epa_katrina_lessons,Lessons Learned: EPA’s Response to Hurricane Katrina,True,"Communication is a key component of emergency response. In two instances brought to our attention, EPA visited two municipal wastewater facilities, asked if their facilities ne..."
4,P005,5,Tier 1 | Tier 1 | Tier 1 | Tier 1 | Tier 1,1,Tier 1,True,epa_katrina_lessons,Lessons Learned: EPA’s Response to Hurricane Katrina,True,EPA regional officials concurred that tremendous communication issues existed following Hurricane Katrina. Both Regions 4 and 6 indicated they have taken a number of steps to i...
5,P006,5,Tier 4 | Tier 4 | Tier 4 | Tier 4 | Tier 4,1,Tier 4,True,epa_katrina_lessons,Lessons Learned: EPA’s Response to Hurricane Katrina,True,"This memorandum addresses the fifth and final question of our drinking water evaluation – whether EPA followed its emergency response protocols, including those lessons learned..."
6,P007,5,Tier 4 | Tier 4 | Tier 4 | Tier 4 | Tier 4,1,Tier 4,True,epa_katrina_lessons,Lessons Learned: EPA’s Response to Hurricane Katrina,True,Regional officials told us that the primary reasons the response to restoring drinking water supplies worked well were the good working relationships and collaborative planning...


## 16. Build the human-adjudication priority queue and workbook

In [30]:
def phase1_definition_signals() -> pd.DataFrame:
    data = SCORED["phase1"].copy()
    definition_data = data[data["definition_id"].isin([
        "D1_old_broad", "D2_current_strict", "D3_provisional_adaptive"
    ])]
    signals = (
        definition_data.groupby(["row_id", "definition_id"])
        .agg(
            yes_share=("pred_int", "mean"),
            model_predictions=("pred_int", "size"),
            evidence_valid_share=("evidence_valid_normalized", "mean"),
        )
        .reset_index()
        .pivot(index="row_id", columns="definition_id")
    )
    signals.columns = [f"{metric}__{definition_id}" for metric, definition_id in signals.columns]
    signals = signals.reset_index()
    yes_columns = [column for column in signals if column.startswith("yes_share__")]
    signals["definition_prediction_range"] = signals[yes_columns].max(axis=1) - signals[yes_columns].min(axis=1)
    signals["definition_disagreement"] = signals["definition_prediction_range"].gt(0)
    return signals


DEFINITION_SIGNALS = phase1_definition_signals()

all_phase1_votes = (
    SCORED["phase1"]
    .groupby("row_id")
    .agg(
        phase1_yes_share=("pred_int", "mean"),
        phase1_all_no=("pred_int", lambda x: bool(pd.Series(x).eq(0).all())),
        phase1_all_yes=("pred_int", lambda x: bool(pd.Series(x).eq(1).all())),
        phase1_evidence_valid_share=("evidence_valid_normalized", "mean"),
    )
    .reset_index()
)

ADJUDICATION_QUEUE = (
    BENCHMARK
    .merge(DEFINITION_SIGNALS, on="row_id", how="left", validate="one_to_one")
    .merge(all_phase1_votes, on="row_id", how="left", validate="one_to_one")
    .merge(
        TIER_STABILITY[["row_id", "modal_tier", "tier_stable", "tiers_by_seed"]],
        on="row_id",
        how="left",
        validate="one_to_one",
    )
)

ADJUDICATION_QUEUE["historical_int"] = ADJUDICATION_QUEUE["gold_historical"].map(label_to_int)
ADJUDICATION_QUEUE["unanimous_phase1_historical_disagreement"] = (
    (ADJUDICATION_QUEUE["historical_int"].eq(1) & ADJUDICATION_QUEUE["phase1_all_no"])
    | (ADJUDICATION_QUEUE["historical_int"].eq(0) & ADJUDICATION_QUEUE["phase1_all_yes"])
)
ADJUDICATION_QUEUE["priority_score"] = (
    ADJUDICATION_QUEUE["is_epa"].fillna(False).astype(int) * 3
    + ADJUDICATION_QUEUE["definition_disagreement"].fillna(False).astype(int) * 3
    + ADJUDICATION_QUEUE["unanimous_phase1_historical_disagreement"].fillna(False).astype(int) * 3
    + (~ADJUDICATION_QUEUE["tier_stable"].fillna(True)).astype(int) * 2
    + ADJUDICATION_QUEUE["phase1_evidence_valid_share"].fillna(1).lt(1).astype(int)
)
ADJUDICATION_QUEUE["priority_band"] = pd.cut(
    ADJUDICATION_QUEUE["priority_score"],
    bins=[-1, 2, 5, 8, np.inf],
    labels=["Routine", "Medium", "High", "Critical"],
)
ADJUDICATION_QUEUE = ADJUDICATION_QUEUE.sort_values(
    ["priority_score", "is_epa", "row_id"],
    ascending=[False, False, True],
)

display(ADJUDICATION_QUEUE[[
    "row_id", "document_id", "is_epa", "gold_historical",
    "definition_disagreement", "unanimous_phase1_historical_disagreement",
    "modal_tier", "tier_stable", "priority_score", "priority_band",
]].head(30))

,row_id,document_id,is_epa,gold_historical,definition_disagreement,unanimous_phase1_historical_disagreement,modal_tier,tier_stable,priority_score,priority_band
8,P009,epa_katrina_lessons,True,Yes,True,False,Tier 3,False,8,High
13,P014,epa_katrina_lessons,True,Yes,True,False,Tier 3,False,8,High
1,P002,epa_katrina_lessons,True,No,True,False,Tier 3,True,7,High
2,P003,epa_katrina_lessons,True,Yes,True,False,Tier 3,True,7,High
3,P004,epa_katrina_lessons,True,Yes,True,False,Tier 2,True,7,High
9,P010,epa_katrina_lessons,True,Yes,True,False,Tier 3,True,7,High
10,P011,epa_katrina_lessons,True,Yes,True,False,Tier 2,True,7,High
16,P017,epa_katrina_lessons,True,Yes,True,False,Tier 1,True,7,High
0,P001,epa_katrina_lessons,True,No,True,False,Tier 4,True,6,High
4,P005,epa_katrina_lessons,True,Yes,True,False,Tier 1,True,6,High


In [31]:
from openpyxl import load_workbook
from openpyxl.formatting.rule import ColorScaleRule
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.worksheet.datavalidation import DataValidation

ADJUDICATION_TEMPLATE_PATH = ADJUDICATION_ROOT / "adjudication_template.xlsx"

editable_columns = [
    "row_id",
    "document_id",
    "document_title",
    "is_epa",
    "pdf_page",
    "section_heading",
    "target_text",
    "context_previous_2",
    "context_previous_1",
    "context_next_1",
    "context_next_2",
    "gold_historical",
    "gold_old_definition",
    "gold_current_definition",
    "gold_final_definition",
    "gold_target",
    "gold_agency",
    "adjudication_status",
    "adjudication_notes",
    "priority_score",
    "priority_band",
    "definition_disagreement",
    "unanimous_phase1_historical_disagreement",
    "modal_tier",
    "tier_stable",
]

for column in editable_columns:
    if column not in ADJUDICATION_QUEUE.columns:
        ADJUDICATION_QUEUE[column] = None

adjudication_sheet = ADJUDICATION_QUEUE[editable_columns].copy()
if "adjudication_status" not in BENCHMARK.columns:
    adjudication_sheet["adjudication_status"] = "Pending"
else:
    adjudication_sheet["adjudication_status"] = adjudication_sheet["adjudication_status"].fillna("Pending")

instructions = pd.DataFrame({
    "Step": [1, 2, 3, 4, 5, 6],
    "Instruction": [
        "Review rows in Critical/High priority order; model predictions are signals, not gold labels.",
        "Code each row independently under the old definition, current definition, and candidate final definition.",
        "Use only Yes or No in the three gold-definition columns.",
        "Resolve the target taxonomy separately; use exactly one canonical target category for final-positive rows.",
        "Do not alter gold_historical; it is retained for provenance.",
        "When complete, save a copy as adjudication_completed.xlsx in this same Drive folder and rerun the analysis notebook.",
    ],
})

with pd.ExcelWriter(ADJUDICATION_TEMPLATE_PATH, engine="openpyxl") as writer:
    instructions.to_excel(writer, sheet_name="Read Me", index=False)
    adjudication_sheet.to_excel(writer, sheet_name="Adjudication", index=False)
    SOURCE_DEFINITIONS.to_excel(writer, sheet_name="Definitions", index=False)
    DEFINITION_SIGNALS.to_excel(writer, sheet_name="Model Signals", index=False)
    TARGET_TAXONOMY_AUDIT.to_excel(writer, sheet_name="Target Audit", index=False)
    DOCUMENT_REGISTRY.to_excel(writer, sheet_name="Document Registry", index=False)

workbook = load_workbook(ADJUDICATION_TEMPLATE_PATH)
for worksheet in workbook.worksheets:
    worksheet.freeze_panes = "A2"
    worksheet.auto_filter.ref = worksheet.dimensions
    for cell in worksheet[1]:
        cell.font = Font(color="FFFFFF", bold=True)
        cell.fill = PatternFill("solid", fgColor="1F4E78")
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    for column_cells in worksheet.columns:
        max_length = max(
            len(str(cell.value)) if cell.value is not None else 0
            for cell in list(column_cells)[:200]
        )
        worksheet.column_dimensions[column_cells[0].column_letter].width = min(max(max_length + 2, 11), 60)
    for row in worksheet.iter_rows(min_row=2):
        for cell in row:
            cell.alignment = Alignment(vertical="top", wrap_text=True)

adjudication_ws = workbook["Adjudication"]
headers = {cell.value: cell.column_letter for cell in adjudication_ws[1]}
label_validation = DataValidation(type="list", formula1='"Yes,No"', allow_blank=True)
status_validation = DataValidation(
    type="list",
    formula1='"Pending,Reviewed,Adjudicated,Needs discussion,Excluded"',
    allow_blank=False,
)
target_validation = DataValidation(
    type="list",
    formula1='"leadership,laws_plans_policies,capabilities,funds_resources,misc_organizational,none"',
    allow_blank=True,
)
adjudication_ws.add_data_validation(label_validation)
adjudication_ws.add_data_validation(status_validation)
adjudication_ws.add_data_validation(target_validation)
for column in ["gold_old_definition", "gold_current_definition", "gold_final_definition"]:
    letter = headers[column]
    label_validation.add(f"{letter}2:{letter}{adjudication_ws.max_row}")
status_letter = headers["adjudication_status"]
status_validation.add(f"{status_letter}2:{status_letter}{adjudication_ws.max_row}")
target_letter = headers["gold_target"]
target_validation.add(f"{target_letter}2:{target_letter}{adjudication_ws.max_row}")
priority_letter = headers["priority_score"]
adjudication_ws.conditional_formatting.add(
    f"{priority_letter}2:{priority_letter}{adjudication_ws.max_row}",
    ColorScaleRule(
        start_type="min", start_color="63BE7B",
        mid_type="percentile", mid_value=50, mid_color="FFEB84",
        end_type="max", end_color="F8696B",
    ),
)
workbook.save(ADJUDICATION_TEMPLATE_PATH)

print("Adjudication template:", ADJUDICATION_TEMPLATE_PATH)
print("Completed-file location expected on a later run:", ADJUDICATION_COMPLETED_PATH)

Adjudication template: /content/drive/MyDrive/Unlearning_Project/prelangchain_ab_v1_2_workspace/analysis/prelangchain_posthoc_v1_3_1/adjudication/adjudication_template.xlsx
Completed-file location expected on a later run: /content/drive/MyDrive/Unlearning_Project/prelangchain_ab_v1_2_workspace/analysis/prelangchain_posthoc_v1_3_1/adjudication/adjudication_completed.xlsx


## 17. Corrected selection interpretation and candidate set

In [32]:
def phase_candidate_table(phase: str, ranking: pd.DataFrame) -> pd.DataFrame:
    if ranking.empty:
        return pd.DataFrame()
    status = selection_status(phase, ranking)
    candidates = ranking[
        ranking["eligible"] | ranking["pareto_front"] | ranking["rank"].le(3)
    ].copy()
    candidates["selection_status"] = status["status"]
    candidates["selection_basis"] = status["selection_basis"]
    candidates["provisional_candidate"] = candidates["condition_id"].eq(
        status.get("selected_condition_id") or status.get("minimum_shortfall_candidate")
    )
    return candidates


PHASE1_CANDIDATES = phase_candidate_table("phase1", PHASE1_RANKING)
PHASE2_CANDIDATES = phase_candidate_table("phase2", PHASE2_RANKING)
PHASE3_CANDIDATES = phase_candidate_table("phase3", PHASE3_RANKING)

CORRECTED_SELECTION_SUMMARY = {
    "created_at_utc": utc_now_iso(),
    "analysis_version": ANALYSIS_VERSION,
    "source_protocol": SOURCE_PROTOCOL,
    "gold_basis": COMMON_FINAL_BASIS,
    "final_gold_complete": FINAL_GOLD_COMPLETE,
    "source_selections_preserved": SOURCE_SELECTIONS,
    "corrected_selection_status": SELECTION_STATUS.to_dict("records"),
    "interpretation": (
        "No source selection file was overwritten. A final winner is declared only "
        "when final adjudicated gold is complete and a condition meets all preregistered constraints."
    ),
}

atomic_write_text(
    REPORTS_ROOT / "corrected_selection_summary.json",
    json.dumps(CORRECTED_SELECTION_SUMMARY, indent=2, default=str),
)

display(PHASE1_CANDIDATES)
display(PHASE2_CANDIDATES)
display(PHASE3_CANDIDATES)

,condition_id,definition_id,prompt_style,context_id,workflow,providers,mean_accuracy,mean_balanced_accuracy,mean_precision_yes,mean_recall_yes,mean_specificity,mean_f1_yes,mean_mcc,mean_kappa,schema_valid_rate,evidence_valid_strict_rate,evidence_valid_rate,total_cost_usd,mean_latency_seconds,worst_provider_document_recall,eligible,constraint_shortfall,pareto_front,rank,selection_status,selection_basis,provisional_candidate
0,P1_DIRECT,D0_none,P1_direct,C1_target,W1_one_stage,3,0.468254,0.616667,0.933333,0.333333,0.900000,0.485364,0.230461,0.135515,1.0,0.952381,0.976190,0.300032,1.810204,0.166667,False,0.003810,True,1,provisional_historical_gold,provisional minimum-shortfall candidate; not a final winner,True
1,P3_D2_CURRENT,D2_current_strict,P3_evidence_checklist,C1_target,W1_one_stage,3,0.507937,0.665625,0.979167,0.364583,0.966667,0.525794,0.316138,0.197925,1.0,0.896825,0.944444,0.425127,2.307064,0.187500,False,0.035556,True,2,provisional_historical_gold,provisional minimum-shortfall candidate; not a final winner,False
2,P3_D3_ADAPTIVE,D3_provisional_adaptive,P3_evidence_checklist,C1_target,W1_one_stage,3,0.380952,0.582292,0.972222,0.197917,0.966667,0.311146,0.194342,0.091408,1.0,0.904762,0.944444,0.465131,2.398845,0.000000,False,0.035556,False,3,provisional_historical_gold,provisional minimum-shortfall candidate; not a final winner,False
3,P2_D3_ADAPTIVE,D3_provisional_adaptive,P2_simple_definition,C1_target,W1_one_stage,3,0.404762,0.586458,0.952381,0.239583,0.933333,0.364389,0.194773,0.097348,1.0,0.912698,0.944444,0.373258,1.940786,0.000000,False,0.035556,True,4,provisional_historical_gold,provisional minimum-shortfall candidate; not a final winner,False
4,P2_D2_CURRENT,D2_current_strict,P2_simple_definition,C1_target,W1_one_stage,3,0.428571,0.613542,0.969697,0.260417,0.966667,0.408030,0.242339,0.126015,1.0,0.904762,0.920635,0.325021,2.031538,0.062500,False,0.059365,True,5,provisional_historical_gold,provisional minimum-shortfall candidate; not a final winner,False
5,P3_D1_OLD,D1_old_broad,P3_evidence_checklist,C1_target,W1_one_stage,3,0.555556,0.605208,0.847547,0.510417,0.700000,0.635454,0.180975,0.145493,1.0,0.888889,0.944444,0.428266,2.448193,0.400000,False,0.135556,True,6,provisional_historical_gold,provisional minimum-shortfall candidate; not a final winner,False
6,P2_D1_OLD,D1_old_broad,P2_simple_definition,C1_target,W1_one_stage,3,0.571429,0.604167,0.842029,0.541667,0.666667,0.652795,0.180926,0.151540,1.0,0.912698,0.936508,0.343165,1.941406,0.333333,False,0.176825,True,7,provisional_historical_gold,provisional minimum-shortfall candidate; not a final winner,False


,condition_id,definition_id,prompt_style,context_id,workflow,providers,mean_accuracy,mean_balanced_accuracy,mean_precision_yes,mean_recall_yes,mean_specificity,mean_f1_yes,mean_mcc,mean_kappa,schema_valid_rate,evidence_valid_strict_rate,evidence_valid_rate,total_cost_usd,mean_latency_seconds,worst_provider_document_recall,eligible,constraint_shortfall,pareto_front,rank,selection_status,selection_basis,provisional_candidate
0,P1_DIRECT__C1_target,D0_none,P1_direct,C1_target,W1_one_stage,3,0.468254,0.616667,0.933333,0.333333,0.900000,0.485364,0.230461,0.135515,1.0,0.952381,0.976190,0.300032,1.810204,0.166667,False,0.003810,True,1,provisional_historical_gold,provisional minimum-shortfall candidate; not a final winner,True
1,P1_DIRECT__C2_metadata,D0_none,P1_direct,C2_metadata,W1_one_stage,3,0.492063,0.609375,0.903704,0.385417,0.833333,0.531590,0.208071,0.131801,1.0,0.936508,0.952381,0.323340,1.888263,0.166667,False,0.027619,True,2,provisional_historical_gold,provisional minimum-shortfall candidate; not a final winner,False
2,P3_D2_CURRENT__C3_plusminus1,D2_current_strict,P3_evidence_checklist,C3_plusminus1,W1_one_stage,3,0.555556,0.696875,0.984127,0.427083,0.966667,0.580845,0.364237,0.252842,1.0,0.880952,0.952381,0.496904,2.625335,0.125000,False,0.027619,True,3,provisional_historical_gold,provisional minimum-shortfall candidate; not a final winner,False
3,P3_D2_CURRENT__C2_metadata,D2_current_strict,P3_evidence_checklist,C2_metadata,W1_one_stage,3,0.500000,0.660417,0.980392,0.354167,0.966667,0.507888,0.310029,0.193734,1.0,0.904762,0.952381,0.434097,2.571467,0.062500,False,0.027619,True,4,provisional_historical_gold,provisional minimum-shortfall candidate; not a final winner,False
4,P3_D2_CURRENT__C1_target,D2_current_strict,P3_evidence_checklist,C1_target,W1_one_stage,3,0.507937,0.665625,0.979167,0.364583,0.966667,0.525794,0.316138,0.197925,1.0,0.896825,0.944444,0.425127,2.307064,0.187500,False,0.035556,True,5,provisional_historical_gold,provisional minimum-shortfall candidate; not a final winner,False
5,P1_DIRECT__C3_plusminus1,D0_none,P1_direct,C3_plusminus1,W1_one_stage,3,0.571429,0.661458,0.935897,0.489583,0.833333,0.626096,0.295293,0.207342,1.0,0.896825,0.928571,0.394790,2.113408,0.333333,False,0.051429,True,6,provisional_historical_gold,provisional minimum-shortfall candidate; not a final winner,False
6,P3_D2_CURRENT__C4_plusminus2,D2_current_strict,P3_evidence_checklist,C4_plusminus2,W1_one_stage,3,0.571429,0.707292,0.985507,0.447917,0.966667,0.594826,0.382743,0.275993,1.0,0.841270,0.928571,0.539600,2.991171,0.187500,False,0.051429,True,7,provisional_historical_gold,provisional minimum-shortfall candidate; not a final winner,False
7,P1_DIRECT__C4_plusminus2,D0_none,P1_direct,C4_plusminus2,W1_one_stage,3,0.563492,0.621875,0.896296,0.510417,0.733333,0.617739,0.230519,0.166650,1.0,0.873016,0.936508,0.452722,2.289421,0.250000,False,0.110159,True,8,provisional_historical_gold,provisional minimum-shortfall candidate; not a final winner,False


,condition_id,definition_id,prompt_style,context_id,workflow,providers,mean_accuracy,mean_balanced_accuracy,mean_precision_yes,mean_recall_yes,mean_specificity,mean_f1_yes,mean_mcc,mean_kappa,schema_valid_rate,evidence_valid_strict_rate,evidence_valid_rate,total_cost_usd,mean_latency_seconds,worst_provider_document_recall,eligible,constraint_shortfall,pareto_front,rank,selection_status,selection_basis,provisional_candidate
0,W1_ONE_STAGE,D0_none,P1_direct,C1_target,W1_one_stage,3,0.47619,0.621875,0.933333,0.343750,0.900000,0.498372,0.237386,0.142209,1.0,0.936508,0.960317,0.301246,1.813200,0.166667,False,0.019683,True,1,provisional_historical_gold,provisional minimum-shortfall candidate; not a final winner,True
1,W2_BINARY_FIRST,D0_none,P1_direct,C1_target,W2_binary_first,3,0.47619,0.598958,0.895833,0.364583,0.833333,0.516414,0.190022,0.114546,1.0,0.896825,0.936508,0.353014,2.111484,0.166667,False,0.043492,True,2,provisional_historical_gold,provisional minimum-shortfall candidate; not a final winner,False


## 18. Targeted confirmatory experiment plan — generated, not executed

In [33]:
def select_confirmatory_context_candidates(ranking: pd.DataFrame, n: int = 2) -> list[str]:
    if ranking.empty:
        return []
    preferred = ranking[
        ranking["pareto_front"]
        & ranking["context_id"].isin(["C3_plusminus1", "C4_plusminus2"])
    ]
    if preferred.empty:
        preferred = ranking[ranking["context_id"].isin(["C3_plusminus1", "C4_plusminus2"])]
    return preferred.head(n)["condition_id"].tolist()


CONFIRMATORY_CONTEXT_CANDIDATES = select_confirmatory_context_candidates(PHASE2_RANKING, n=2)
CONFIRMATORY_PLAN = {
    "status": "deferred until adjudication and corrected rescoring are complete",
    "created_at_utc": utc_now_iso(),
    "purpose": "Rerun target-only and the strongest context candidate contemporaneously to remove the C1 timing confound.",
    "candidate_context_conditions": CONFIRMATORY_CONTEXT_CANDIDATES,
    "required_control": "matched C1 target-only condition using the same definition and prompt structure",
    "providers": list(EXPECTED_PROVIDERS),
    "suggested_repeats": [17, 43, 101],
    "randomization": "randomize condition order within row and provider",
    "one_paragraph_per_request": True,
    "new_api_calls_launched_by_this_notebook": False,
}
atomic_write_text(
    REPORTS_ROOT / "confirmatory_experiment_plan.json",
    json.dumps(CONFIRMATORY_PLAN, indent=2),
)
print(json.dumps(CONFIRMATORY_PLAN, indent=2))

{
  "status": "deferred until adjudication and corrected rescoring are complete",
  "created_at_utc": "2026-07-24T00:46:03.932657+00:00",
  "purpose": "Rerun target-only and the strongest context candidate contemporaneously to remove the C1 timing confound.",
  "candidate_context_conditions": [
    "P3_D2_CURRENT__C3_plusminus1",
    "P1_DIRECT__C3_plusminus1"
  ],
  "required_control": "matched C1 target-only condition using the same definition and prompt structure",
  "providers": [
    "openai",
    "anthropic",
    "gemini"
  ],
  "suggested_repeats": [
    17,
    43,
    101
  ],
  "randomization": "randomize condition order within row and provider",
  "one_paragraph_per_request": true,
  "new_api_calls_launched_by_this_notebook": false
}


## 19. Export corrected tables and report workbook

In [34]:
SCORED_ROOT = ANALYSIS_ROOT / "scored"
SCORED_ROOT.mkdir(parents=True, exist_ok=True)
SCORED_DATASET_MANIFEST_ROWS = []
for phase_name, phase_frame in SCORED.items():
    scored_path = SCORED_ROOT / f"{phase_name}_corrected_scored.csv.gz"
    phase_frame.to_csv(scored_path, index=False, compression="gzip")
    SCORED_DATASET_MANIFEST_ROWS.append({
        "dataset": phase_name,
        "filename": scored_path.name,
        "rows": len(phase_frame),
        "columns": len(phase_frame.columns),
        "sha256": sha256_file(scored_path),
    })
SCORED_DATASET_MANIFEST = pd.DataFrame(SCORED_DATASET_MANIFEST_ROWS)
SCORED_DATASET_MANIFEST.to_csv(
    MANIFEST_ROOT / "scored_dataset_manifest.csv",
    index=False,
)

EXPORT_TABLES: dict[str, pd.DataFrame] = {
    "Configuration": pd.DataFrame([CONFIGURATION]),
    "Snapshot Manifest": SNAPSHOT_MANIFEST,
    "Snapshot Verification": SNAPSHOT_VERIFICATION,
    "Required Artifacts": REQUIRED_ARTIFACT_AUDIT,
    "Import Summary": IMPORT_SUMMARY,
    "Source Selections": pd.DataFrame([
        {"phase": key, **(value if isinstance(value, dict) else {"value": value})}
        for key, value in SOURCE_SELECTIONS.items()
    ]),
    "Source Execution Summary": pd.DataFrame([SOURCE_EXECUTION_SUMMARY]),
    "Gold Availability": GOLD_AVAILABILITY,
    "Document Registry": DOCUMENT_REGISTRY,
    "EPA Match Audit": EPA_MATCH_AUDIT,
    "Corrected Completeness": CORRECTED_COMPLETENESS,
    "Completeness Summary": COMPLETENESS_SUMMARY,
    "Evidence Failure Summary": EVIDENCE_FAILURE_SUMMARY,
    "Evidence Quotes": EVIDENCE_QUOTES,
    "Phase1 Ranking": PHASE1_RANKING,
    "Phase1 Provider": PHASE1_PROVIDER_METRICS,
    "Phase1 Documents": PHASE1_DOCUMENT_METRICS,
    "Phase1 Aligned": PHASE1_ALIGNED_METRICS,
    "Phase1 Tradeoff": PHASE1_DEFINITION_TRADEOFF,
    "Phase1 EPA Exact": PHASE1_EPA_DIAGNOSTICS,
    "Definition Impact": DEFINITION_IMPACT,
    "D3 Boundary Cases": D3_BOUNDARY_CASES,
    "Phase2 Ranking": PHASE2_RANKING,
    "Phase2 Provider": PHASE2_PROVIDER_METRICS,
    "Phase2 Documents": PHASE2_DOCUMENT_METRICS,
    "Phase2 Timing": PHASE2_TIMING_AUDIT,
    "Phase2 Context Comparison": PHASE2_CONTEXT_COMPARISON,
    "Phase3 Ranking": PHASE3_RANKING,
    "Phase3 Provider": PHASE3_PROVIDER_METRICS,
    "Phase3 Documents": PHASE3_DOCUMENT_METRICS,
    "Phase3 Target Audit": PHASE3_TARGET_METRICS,
    "Target Per Category": TARGET_PER_CATEGORY_RECALL,
    "Scored Dataset Manifest": SCORED_DATASET_MANIFEST,
    "Target Taxonomy": TARGET_TAXONOMY_AUDIT,
    "Target Taxonomy Summary": TARGET_TAXONOMY_SUMMARY,
    "Stability Reliability": STABILITY_RELIABILITY,
    "Provider Row Stability": PROVIDER_ROW_STABILITY,
    "Provider Seed Pairwise": PROVIDER_SEED_PAIRWISE,
    "Tiered Review": TIERED_REVIEW_BY_SEED,
    "Tier Stability": TIER_STABILITY,
    "Adjudication Queue": ADJUDICATION_QUEUE,
    "Selection Status": SELECTION_STATUS,
    "Phase1 Candidates": PHASE1_CANDIDATES,
    "Phase2 Candidates": PHASE2_CANDIDATES,
    "Phase3 Candidates": PHASE3_CANDIDATES,
}
EXPORT_TABLES = {
    name: frame
    for name, frame in EXPORT_TABLES.items()
    if isinstance(frame, pd.DataFrame) and not frame.empty
}

TABLE_MANIFEST_ROWS = []
for name, frame in EXPORT_TABLES.items():
    filename = re.sub(r"[^A-Za-z0-9_-]+", "_", name).strip("_").lower() + ".csv"
    path = TABLES_ROOT / filename
    frame.to_csv(path, index=False)
    TABLE_MANIFEST_ROWS.append({
        "table": name,
        "filename": filename,
        "rows": len(frame),
        "columns": len(frame.columns),
        "sha256": sha256_file(path),
    })

TABLE_MANIFEST = pd.DataFrame(TABLE_MANIFEST_ROWS)
TABLE_MANIFEST.to_csv(MANIFEST_ROOT / "table_manifest.csv", index=False)
display(TABLE_MANIFEST)

,table,filename,rows,columns,sha256
0,Configuration,configuration.csv,1,11,6a8185620b279d6c870bd9e02ff7f20cd3ae12ef6e5d64331182eb600af2c616
1,Snapshot Manifest,snapshot_manifest.csv,122,7,bff26a6de565c5acafe79bbe476b700737e3d5884c5466becab56bbd726215fc
2,Snapshot Verification,snapshot_verification.csv,122,5,4901fdc336e080717bb5324ef356d5418a79397cd55980481036bb96104f5e16
3,Required Artifacts,required_artifacts.csv,18,3,af01a879f3b19ea955b543e724498703a53425bee5fcdae0f19450e60c66d69d
4,Import Summary,import_summary.csv,4,5,1ca3c7b2a843616b78137d48d74a82bc59c0f9aed92332efd2ab45092dfe8b4c
5,Source Selections,source_selections.csv,4,34,7f87bff3aff1e1701303fb49dd09dca5d0ea6e3f2a058aa446f96437844e064c
6,Source Execution Summary,source_execution_summary.csv,1,17,635b7ab1ffa92e445bfbe49e266aa4cb74f458beb36a1136a784c60d4ada86f2
7,Gold Availability,gold_availability.csv,4,3,6499a22f5ee78908c92cc76ca0c41fb073d0a1e2390d3c36c704fef56242134e
8,Document Registry,document_registry.csv,3,3,fd65c689de636533f828b2bcf211265629e4eb44b7badae2c656d88ece78e14e
9,EPA Match Audit,epa_match_audit.csv,42,6,5865e8e208d2c47468e1ddce05fdacb20b4988d4e7363ab102511b3e0b0c1ed1


In [35]:
def unique_sheet_name(name: str, used: set[str]) -> str:
    base = re.sub(r"[\/*?:\[\]]", "_", name)[:31] or "Sheet"
    candidate = base
    counter = 1
    while candidate in used:
        suffix = f"_{counter}"
        candidate = base[:31 - len(suffix)] + suffix
        counter += 1
    used.add(candidate)
    return candidate


def excel_safe_dataframe(frame: pd.DataFrame) -> pd.DataFrame:
    """Return a copy that Excel can serialize without losing UTC meaning."""
    safe = frame.copy()
    for column in safe.columns:
        series = safe[column]
        if isinstance(series.dtype, pd.DatetimeTZDtype):
            safe[column] = series.dt.tz_convert("UTC").dt.tz_localize(None)
            continue
        if pd.api.types.is_datetime64_any_dtype(series.dtype):
            continue
        if series.dtype == object:
            def convert_value(value: Any) -> Any:
                if isinstance(value, pd.Timestamp):
                    if value.tzinfo is not None:
                        return value.tz_convert("UTC").tz_localize(None)
                    return value
                if isinstance(value, datetime):
                    if value.tzinfo is not None:
                        return value.astimezone(timezone.utc).replace(tzinfo=None)
                    return value
                if isinstance(value, (dict, list, tuple, set)):
                    return canonical_json(value)
                return value
            safe[column] = series.map(convert_value)
    return safe


def format_excel_worksheet(worksheet) -> None:
    from openpyxl.styles import Alignment, Font, PatternFill
    from openpyxl.utils import get_column_letter
    worksheet.freeze_panes = "A2"
    worksheet.auto_filter.ref = worksheet.dimensions
    header_fill = PatternFill("solid", fgColor="1F4E78")
    header_font = Font(color="FFFFFF", bold=True)
    for cell in worksheet[1]:
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    for column_cells in worksheet.columns:
        letter = get_column_letter(column_cells[0].column)
        max_length = max(
            len(str(cell.value)) if cell.value is not None else 0
            for cell in list(column_cells)[:200]
        )
        worksheet.column_dimensions[letter].width = min(max(max_length + 2, 10), 60)
    for row in worksheet.iter_rows(min_row=2):
        for cell in row:
            cell.alignment = Alignment(vertical="top", wrap_text=True)


CORRECTED_REPORT_WORKBOOK = REPORTS_ROOT / "prelangchain_posthoc_corrected_report.xlsx"
with pd.ExcelWriter(CORRECTED_REPORT_WORKBOOK, engine="openpyxl") as writer:
    used_names: set[str] = set()
    for name, frame in EXPORT_TABLES.items():
        excel_safe_dataframe(frame).to_excel(
            writer,
            sheet_name=unique_sheet_name(name, used_names),
            index=False,
        )
    for worksheet in writer.book.worksheets:
        format_excel_worksheet(worksheet)

print("Corrected report workbook:", CORRECTED_REPORT_WORKBOOK)

Corrected report workbook: /content/drive/MyDrive/Unlearning_Project/prelangchain_ab_v1_2_workspace/analysis/prelangchain_posthoc_v1_3_1/reports/prelangchain_posthoc_corrected_report.xlsx


## 20. Analysis manifest and self-tests

In [36]:
def manifest_directory(root: Path) -> pd.DataFrame:
    rows = []
    for path in sorted(root.rglob("*")):
        if path.is_file() and ".partial-" not in path.name:
            rows.append({
                "relative_path": str(path.relative_to(root)),
                "size_bytes": path.stat().st_size,
                "modified_utc": datetime.fromtimestamp(path.stat().st_mtime, tz=timezone.utc).isoformat(),
                "sha256": sha256_file(path),
            })
    return pd.DataFrame(rows)


ANALYSIS_MANIFEST = manifest_directory(ANALYSIS_ROOT)
ANALYSIS_MANIFEST.to_csv(MANIFEST_ROOT / "analysis_artifact_manifest.csv", index=False)

def safe_package_version(package: str) -> Optional[str]:
    try:
        return importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        return None


ENVIRONMENT_SNAPSHOT = {
    "captured_at_utc": utc_now_iso(),
    "python": sys.version,
    "packages": {
        package: safe_package_version(package)
        for package in [
            "numpy", "pandas", "openpyxl", "scikit-learn",
            "scipy", "statsmodels", "krippendorff", "pyarrow",
        ]
    },
}
atomic_write_text(
    MANIFEST_ROOT / "environment_snapshot.json",
    json.dumps(ENVIRONMENT_SNAPSHOT, indent=2),
)

print("Analysis artifacts manifested:", len(ANALYSIS_MANIFEST))

Analysis artifacts manifested: 64


In [37]:
def self_tests() -> pd.DataFrame:
    tests = []

    def add(
        name: str,
        passed: bool,
        observed: Any,
        requirement: str,
        severity: str = "error",
    ) -> None:
        tests.append({
            "test": name,
            "passed": bool(passed),
            "observed": observed,
            "requirement": requirement,
            "severity": severity,
        })

    add("analysis-only guard", RUN_API_CALLS is False, RUN_API_CALLS, "False")
    add("snapshot locked", SNAPSHOT_LOCK_PATH.is_file(), str(SNAPSHOT_LOCK_PATH), "existing lock")
    add(
        "snapshot hashes verified",
        SNAPSHOT_VERIFICATION["verified"].all(),
        int(SNAPSHOT_VERIFICATION["verified"].sum()),
        len(SNAPSHOT_VERIFICATION),
    )
    add(
        "all required source artifacts present",
        REQUIRED_ARTIFACT_AUDIT["passed"].all(),
        int(REQUIRED_ARTIFACT_AUDIT["passed"].sum()),
        len(REQUIRED_ARTIFACT_AUDIT),
    )
    add(
        "all JSONL lines valid",
        JSONL_AUDIT["valid_json"].all(),
        int(JSONL_AUDIT["valid_json"].sum()),
        len(JSONL_AUDIT),
    )
    add(
        "all corrected completeness groups complete",
        CORRECTED_COMPLETENESS["complete"].all(),
        int(CORRECTED_COMPLETENESS["complete"].sum()),
        len(CORRECTED_COMPLETENESS),
    )
    add(
        "exactly one EPA document identity",
        DOCUMENT_REGISTRY["is_epa"].sum() == 1,
        int(DOCUMENT_REGISTRY["is_epa"].sum()),
        1,
    )
    add(
        "document IDs are unique",
        DOCUMENT_REGISTRY["document_id"].is_unique,
        int(DOCUMENT_REGISTRY["document_id"].nunique()),
        len(DOCUMENT_REGISTRY),
    )
    add(
        "EPA substring false positives exposed",
        EPA_MATCH_AUDIT["false_positive_from_substring"].any(),
        int(EPA_MATCH_AUDIT["false_positive_from_substring"].sum()),
        ">=1 diagnostic row in the real benchmark",
        severity="warning",
    )
    add(
        "prompt hash remains immutable",
        dataframe_sha256(BENCHMARK, PROMPT_INPUT_HASH_COLUMNS) == PROMPT_INPUT_SHA256,
        PROMPT_INPUT_SHA256,
        "same before and after adjudication import",
    )
    add(
        "analysis gold hash is current",
        dataframe_sha256(BENCHMARK, GOLD_HASH_COLUMNS) == ANALYSIS_GOLD_LABELS_SHA256,
        ANALYSIS_GOLD_LABELS_SHA256,
        "hash of current analysis labels",
    )
    add(
        "source and analysis gold hashes recorded",
        all(len(value) == 64 for value in [SOURCE_GOLD_LABELS_SHA256, ANALYSIS_GOLD_LABELS_SHA256]),
        {
            "source": SOURCE_GOLD_LABELS_SHA256,
            "analysis": ANALYSIS_GOLD_LABELS_SHA256,
        },
        "two 64-character SHA-256 values",
    )
    add(
        "target parser preserves comma-bearing single label",
        parse_target_set("Laws, plans and policies") == frozenset({"laws_plans_policies"}),
        sorted(parse_target_set("Laws, plans and policies")),
        "one canonical target",
    )
    add(
        "target parser supports historical multi-label cells",
        parse_target_set("Laws, plans and policies; Leadership")
        == frozenset({"laws_plans_policies", "leadership"}),
        sorted(parse_target_set("Laws, plans and policies; Leadership")),
        "two canonical targets",
    )
    add("phase1 rows imported", len(PHASE1) > 0, len(PHASE1), ">0")
    add("phase2 rows imported", len(PHASE2) > 0, len(PHASE2), ">0")
    add("phase3 rows imported", len(PHASE3) > 0, len(PHASE3), ">0")
    add("stability rows imported", len(STABILITY) > 0, len(STABILITY), ">0")
    add(
        "Krippendorff alpha calculated",
        STABILITY_RELIABILITY["krippendorff_alpha"].notna().all(),
        STABILITY_RELIABILITY["krippendorff_alpha"].tolist(),
        "all non-null",
    )
    add(
        "source selection files preserved",
        all(
            (SNAPSHOT_CONFIG_ROOT / f"{phase}_selection.json").is_file()
            for phase in ["phase1", "phase2", "phase3"]
        ),
        list(SOURCE_SELECTIONS),
        "three frozen source selections",
    )
    add(
        "corrected scored datasets exported",
        len(SCORED_DATASET_MANIFEST) == 4
        and all((SCORED_ROOT / filename).is_file() for filename in SCORED_DATASET_MANIFEST["filename"]),
        SCORED_DATASET_MANIFEST["dataset"].tolist(),
        "phase1, phase2, phase3, stability",
    )
    add(
        "corrected report exists",
        CORRECTED_REPORT_WORKBOOK.is_file(),
        str(CORRECTED_REPORT_WORKBOOK),
        "existing workbook",
    )
    add(
        "adjudication template exists",
        ADJUDICATION_TEMPLATE_PATH.is_file(),
        str(ADJUDICATION_TEMPLATE_PATH),
        "existing workbook",
    )
    add(
        "no final winner on incomplete final gold",
        FINAL_GOLD_COMPLETE
        or not SELECTION_STATUS["status"].eq("final_eligible_winner").any(),
        SELECTION_STATUS["status"].tolist(),
        "no final winner unless final gold is complete",
    )
    return pd.DataFrame(tests)


SELF_TESTS = self_tests()
display(SELF_TESTS)

fatal_failures = SELF_TESTS[
    ~SELF_TESTS["passed"] & SELF_TESTS["severity"].eq("error")
]
if not fatal_failures.empty:
    raise AssertionError(
        "Post-hoc notebook self-tests failed:\n"
        + fatal_failures.to_string(index=False)
    )

,test,passed,observed,requirement,severity
0,analysis-only guard,True,False,False,error
1,snapshot locked,True,/content/drive/MyDrive/Unlearning_Project/prelangchain_ab_v1_2_workspace/frozen_runs/prelangchain_v12_discovery_r1_complete/SNAPSHOT_LOCK.json,existing lock,error
2,snapshot hashes verified,True,122,122,error
3,all required source artifacts present,True,18,18,error
4,all JSONL lines valid,True,2578,2578,error
5,all corrected completeness groups complete,True,69,69,error
6,exactly one EPA document identity,True,1,1,error
7,document IDs are unique,True,3,3,error
8,EPA substring false positives exposed,True,21,>=1 diagnostic row in the real benchmark,warning
9,prompt hash remains immutable,True,107cdf61cc742a491c993b1f1eb27d69c108ce9ef359379664eba471270170af,same before and after adjudication import,error


In [38]:
ANALYSIS_SUMMARY = {
    "analysis_version": ANALYSIS_VERSION,
    "completed_at_utc": utc_now_iso(),
    "source_protocol": SOURCE_PROTOCOL,
    "snapshot_root": str(SNAPSHOT_ROOT),
    "analysis_root": str(ANALYSIS_ROOT),
    "prompt_input_sha256": PROMPT_INPUT_SHA256,
    "source_gold_labels_sha256": SOURCE_GOLD_LABELS_SHA256,
    "analysis_gold_labels_sha256": ANALYSIS_GOLD_LABELS_SHA256,
    "final_gold_complete": FINAL_GOLD_COMPLETE,
    "common_final_basis": COMMON_FINAL_BASIS,
    "phase1_rows": len(SCORED["phase1"]),
    "phase2_rows": len(SCORED["phase2"]),
    "phase3_rows": len(SCORED["phase3"]),
    "stability_rows": len(SCORED["stability"]),
    "all_source_groups_complete": bool(CORRECTED_COMPLETENESS["complete"].all()),
    "target_membership_evaluation_ready": TARGET_EVALUATION_READY,
    "target_final_single_label_ready": TARGET_FINAL_SINGLE_LABEL_READY,
    "selection_status": SELECTION_STATUS.to_dict("records"),
    "adjudication_template": str(ADJUDICATION_TEMPLATE_PATH),
    "adjudication_completed_imported": ADJUDICATION_COMPLETED_PATH.is_file(),
    "corrected_report": str(CORRECTED_REPORT_WORKBOOK),
    "self_tests_passed": bool(SELF_TESTS.loc[SELF_TESTS["severity"].eq("error"), "passed"].all()),
    "next_action": (
        "Complete adjudication and rerun for final rescoring."
        if not FINAL_GOLD_COMPLETE
        else "Review eligible/Pareto candidates and preregister the small confirmatory context rerun."
    ),
}
atomic_write_text(
    REPORTS_ROOT / "analysis_summary.json",
    json.dumps(ANALYSIS_SUMMARY, indent=2, default=str),
)
SELF_TESTS.to_csv(TABLES_ROOT / "self_tests.csv", index=False)
pd.DataFrame([ANALYSIS_SUMMARY]).to_csv(
    TABLES_ROOT / "analysis_summary.csv",
    index=False,
)

# Add the final audit outputs to the report workbook after they exist.
with pd.ExcelWriter(
    CORRECTED_REPORT_WORKBOOK,
    engine="openpyxl",
    mode="a",
    if_sheet_exists="replace",
) as writer:
    excel_safe_dataframe(SELF_TESTS).to_excel(
        writer, sheet_name="Self Tests", index=False
    )
    excel_safe_dataframe(pd.DataFrame([ANALYSIS_SUMMARY])).to_excel(
        writer,
        sheet_name="Analysis Summary",
        index=False,
    )
    format_excel_worksheet(writer.book["Self Tests"])
    format_excel_worksheet(writer.book["Analysis Summary"])

# Regenerate the table manifest after late self-test and summary exports.
final_table_rows = []
for table_path in sorted(TABLES_ROOT.glob("*.csv")):
    try:
        table_frame = pd.read_csv(table_path)
        rows_count = len(table_frame)
        columns_count = len(table_frame.columns)
    except Exception:
        rows_count = None
        columns_count = None
    final_table_rows.append({
        "filename": table_path.name,
        "rows": rows_count,
        "columns": columns_count,
        "sha256": sha256_file(table_path),
    })
FINAL_TABLE_MANIFEST = pd.DataFrame(final_table_rows)
FINAL_TABLE_MANIFEST.to_csv(
    MANIFEST_ROOT / "table_manifest.csv",
    index=False,
)

# Regenerate the artifact manifest only after every output has been written.
final_manifest_path = MANIFEST_ROOT / "analysis_artifact_manifest.csv"
FINAL_ANALYSIS_MANIFEST = manifest_directory(ANALYSIS_ROOT)
FINAL_ANALYSIS_MANIFEST = FINAL_ANALYSIS_MANIFEST[
    FINAL_ANALYSIS_MANIFEST["relative_path"].ne(
        str(final_manifest_path.relative_to(ANALYSIS_ROOT))
    )
].copy()
FINAL_ANALYSIS_MANIFEST.to_csv(final_manifest_path, index=False)

print(json.dumps(ANALYSIS_SUMMARY, indent=2, default=str))
print("Final analysis artifacts manifested:", len(FINAL_ANALYSIS_MANIFEST))

{
  "analysis_version": "prelangchain_posthoc_v1_3_1",
  "completed_at_utc": "2026-07-24T00:46:27.639525+00:00",
  "source_protocol": "prelangchain_ab_v1_2_api_hardened",
  "snapshot_root": "/content/drive/MyDrive/Unlearning_Project/prelangchain_ab_v1_2_workspace/frozen_runs/prelangchain_v12_discovery_r1_complete",
  "analysis_root": "/content/drive/MyDrive/Unlearning_Project/prelangchain_ab_v1_2_workspace/analysis/prelangchain_posthoc_v1_3_1",
  "prompt_input_sha256": "107cdf61cc742a491c993b1f1eb27d69c108ce9ef359379664eba471270170af",
  "source_gold_labels_sha256": "d471754373ae679bd04b83a218e1032dc440d63f142d9dd9f187f284993cc1fb",
  "analysis_gold_labels_sha256": "d471754373ae679bd04b83a218e1032dc440d63f142d9dd9f187f284993cc1fb",
  "final_gold_complete": false,
  "common_final_basis": "historical provisional gold",
  "phase1_rows": 882,
  "phase2_rows": 1008,
  "phase3_rows": 252,
  "stability_rows": 630,
  "all_source_groups_complete": true,
  "target_membership_evaluation_ready": f

## 21. Next steps

### Before adjudication is complete

1. Preserve the displayed frozen snapshot directory unchanged.
2. Open `adjudication_template.xlsx` from the analysis Drive folder.
3. Review Critical and High-priority rows first, especially exact EPA rows and D1/D2/D3 boundary cases.
4. Complete the three definition-specific binary labels and target taxonomy fields.
5. Save the completed copy as `adjudication_completed.xlsx` in the same folder.

### After the completed workbook exists

Rerun this notebook from the top. It will import the completed labels, recompute the gold hash, rescore every preserved model prediction, regenerate rankings and reliability tables, and declare a final winner only if the final gold is complete and a configuration satisfies all preregistered constraints.

### Confirmatory experiment

Use the generated `confirmatory_experiment_plan.json` only after corrected rescoring. The recommended design reruns a matched C1 control and the strongest context candidate contemporaneously across all three providers and three repeats. LangChain remains deferred until that small confirmation is complete.